# Ilaria project v6 — English
200-step experimental SwypikOS/Swyp Lang continuation. Requires the verified v5 adapter in Drive. Run cells individually. No Swyp chat tool is registered by this notebook. Compiler-tested scalar code examples and evidence-grounded answers are trained; no repository dump is included. Output directories are protected. Python evaluation is separate from final Go verification.

In [ ]:
import os, sys, subprocess, json, hashlib, platform
from pathlib import Path
os.environ["TORCHDYNAMO_DISABLE"] = "1"
subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], check=True)
import torch
assert torch.cuda.is_available(), "Select a GPU runtime first."
print({"python": platform.python_version(), "torch": torch.__version__, "gpu": torch.cuda.get_device_name(0)})
assert torch.cuda.is_bf16_supported(), "This pilot expects BF16 support."

In [ ]:
# Keep Colab's vendor-provided CUDA/PyTorch installation. Pin the tested Python API surface.
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
    "transformers==5.3.0", "safetensors==0.8.0", "huggingface-hub==1.7.1",
    "tokenizers==0.22.2", "accelerate==1.13.0", "numpy"], check=True)

In [ ]:
SOURCES = {'forge/train_tools.py': '"""English-first BitNet LoRA SFT; single GPU or torchrun DDP.\n\nRequires curated --train/--validation JSONL, a local --llm-dir, and an empty\noutput directory (or explicit --resume). No corpus or model downloads.\nValidation measures assistant-token loss per language, NOT task success.\n"""\nimport os\nos.environ.setdefault("TORCHDYNAMO_DISABLE", "1")\nimport argparse\nimport hashlib\nimport json\nimport math\nfrom pathlib import Path\nimport sys\nimport time\n\nsys.path.insert(0, str(Path(__file__).parent / "multimodal"))\nif __package__:\n    from .tool_data import load_trajectories, validate_splits, encode_trajectory, ENCODING_VERSION\nelse:\n    from tool_data import load_trajectories, validate_splits, encode_trajectory, ENCODING_VERSION\n\n\ndef initialize_adapter(base, prefix, lb, expected_kind, rank, alpha):\n    """Warm start weights only, with strict export validation and provenance."""\n    import torch\n    from safetensors.torch import load_file\n    meta_path, weights_path = Path(prefix + ".json"), Path(prefix + ".safetensors")\n    meta = json.loads(meta_path.read_text(encoding="utf-8"))\n    if meta.get("base", {}).get("kind") != expected_kind:\n        raise ValueError("initial adapter base kind differs")\n    cfg = meta.get("lora", {})\n    modules = lb.lora_modules(base)\n    if (cfg.get("r"), cfg.get("alpha"), cfg.get("scaling"), cfg.get("n_modules")) != (rank, alpha, alpha / rank, len(modules)):\n        raise ValueError("initial adapter LoRA configuration differs")\n    tensors = load_file(str(weights_path), device="cpu")\n    expected = {lb.export_key(name) + "." + key for name in modules for key in ("A", "B")}\n    if set(tensors) != expected:\n        raise ValueError("initial adapter tensor keys differ")\n    state = {}\n    for name, module in modules.items():\n        state[name] = {}\n        for key, parameter in (("A", module.lora_A), ("B", module.lora_B)):\n            value = tensors[lb.export_key(name) + "." + key]\n            if value.shape != parameter.shape or not torch.isfinite(value).all():\n                raise ValueError("invalid initial adapter tensor: " + name + "." + key)\n            state[name][key] = value\n    lb.load_lora_state_dict(base, state)\n    return {"weights_sha256": hashlib.sha256(weights_path.read_bytes()).hexdigest(),\n            "metadata_sha256": hashlib.sha256(meta_path.read_bytes()).hexdigest()}\n\n\ndef main():\n    p = argparse.ArgumentParser(description=__doc__)\n    p.add_argument("--train", required=True)\n    p.add_argument("--validation", required=True)\n    p.add_argument("--languages", default="en")\n    p.add_argument("--llm-dir", required=True)\n    p.add_argument("--out", required=True)\n    p.add_argument("--resume", action="store_true")\n    p.add_argument("--init-adapter", help="Export prefix for a new training stage; fresh optimizer, not exact resume")\n    p.add_argument("--steps", type=int, default=1000)\n    p.add_argument("--batch", type=int, default=1)\n    p.add_argument("--accum", type=int, default=8)\n    p.add_argument("--max-length", type=int, default=2048)\n    p.add_argument("--rank", type=int, default=16)\n    p.add_argument("--alpha", type=int, default=32)\n    p.add_argument("--lr", type=float, default=1e-4)\n    p.add_argument("--checkpoint-every", type=int, default=100)\n    p.add_argument("--seed", type=int, default=42)\n    p.add_argument("--gradient-checkpointing", action="store_true")\n    p.add_argument("--max-runtime-minutes", type=float, default=0, help="Soft training-loop limit; checkpoint and validation take additional time")\n    p.add_argument("--log-every", type=int, default=10)\n    p.add_argument("--validate-only", action="store_true")\n    p.add_argument("--smoke", action="store_true", help="Tiny random CPU model; checks plumbing only, never production weights")\n    args = p.parse_args()\n    for field in ("steps", "batch", "accum", "max_length", "rank", "alpha", "checkpoint_every"):\n        if getattr(args, field) < 1:\n            p.error(f"{field} must be positive")\n    if not math.isfinite(args.lr) or args.lr <= 0:\n        p.error("lr must be finite and positive")\n    if not math.isfinite(args.max_runtime_minutes) or args.max_runtime_minutes < 0 or args.log_every < 1:\n        p.error("runtime limit must be nonnegative and log-every positive")\n    languages = tuple(args.languages.split(","))\n    train = load_trajectories(args.train, languages)\n    validation = load_trajectories(args.validation, languages)\n    counts = validate_splits(train, validation)\n    print(json.dumps(counts, sort_keys=True))\n    if args.validate_only:\n        return\n    os.environ["HF_HUB_OFFLINE"] = "1"\n    import torch\n    import torch.distributed as dist\n    from torch.nn.parallel import DistributedDataParallel\n    from torch.utils.data import DataLoader, DistributedSampler\n    from transformers import AutoTokenizer\n    from safetensors.torch import save_file\n    import lora_bitlinear as lb\n\n    world = int(os.environ.get("WORLD_SIZE", "1"))\n    rank = int(os.environ.get("RANK", "0"))\n    local_rank = int(os.environ.get("LOCAL_RANK", "0"))\n    cuda = torch.cuda.is_available() and not args.smoke\n    device = torch.device("cuda", local_rank) if cuda else torch.device("cpu")\n    if cuda:\n        torch.cuda.set_device(local_rank)\n    if world > 1:\n        dist.init_process_group("nccl" if cuda and os.name != "nt" else "gloo")\n    try:\n        torch.manual_seed(args.seed)\n        if args.smoke:\n            class TinyTokenizer:\n                eos_token_id = 1\n                def encode(self, text, add_special_tokens=False):\n                    return [2 + (ord(c) % 60) for c in text]\n            tokenizer = TinyTokenizer()\n        else:\n            tokenizer = AutoTokenizer.from_pretrained(args.llm_dir, local_files_only=True)\n        encoded = [encode_trajectory(r, tokenizer, args.max_length) for r in train]\n        evaluated = [encode_trajectory(r, tokenizer, args.max_length) for r in validation]\n        if args.smoke:\n            from mm_model import build_tiny_bitnet_llm\n            base, _ = build_tiny_bitnet_llm(16, 64, 0, 1, layers=1, heads=2, kv_heads=1, ffn=32, max_pos=args.max_length)\n            base.requires_grad_(False)\n        else:\n            base = lb.load_frozen_base("offline", args.llm_dir)\n        base = base.to(device)\n        params = lb.inject_lora(base, r=args.rank, alpha=args.alpha, dropout=0.05)\n        base.to(device)\n        initialization = None\n        if args.init_adapter:\n            initialization = initialize_adapter(base, args.init_adapter, lb,\n                "smoke" if args.smoke else "offline", args.rank, args.alpha)\n        base.config.use_cache = False\n        if args.gradient_checkpointing:\n            base.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})\n        model = DistributedDataParallel(base, device_ids=[local_rank] if cuda else None) if world > 1 else base\n        opt = torch.optim.AdamW(params, lr=args.lr)\n        dtype = torch.bfloat16 if cuda and torch.cuda.is_bf16_supported() else torch.float16\n        scaler = torch.amp.GradScaler("cuda", enabled=cuda and dtype == torch.float16)\n        pad = tokenizer.eos_token_id\n        if pad is None:\n            raise ValueError("tokenizer requires an EOS token")\n\n        def collate(rows):\n            n = max(len(r["input_ids"]) for r in rows)\n            return {\n                "input_ids": torch.tensor([r["input_ids"] + [pad] * (n - len(r["input_ids"])) for r in rows], device=device),\n                "labels": torch.tensor([r["labels"] + [-100] * (n - len(r["labels"])) for r in rows], device=device),\n                "attention_mask": torch.tensor([[1] * len(r["input_ids"]) + [0] * (n - len(r["input_ids"])) for r in rows], device=device),\n            }\n\n        signature = hashlib.sha256(json.dumps({"train": train, "validation": validation}, sort_keys=True, ensure_ascii=False).encode()).hexdigest()\n        contract = {k: getattr(args, k) for k in ("llm_dir", "languages", "batch", "accum", "max_length", "rank", "alpha", "lr", "seed", "steps")}\n        contract.update(dataset_sha256=signature, world_size=world, smoke=args.smoke)\n        contract["gradient_checkpointing"] = args.gradient_checkpointing\n        contract["encoding_version"] = ENCODING_VERSION\n        if initialization is not None:\n            contract["initial_adapter"] = initialization\n        out = Path(args.out)\n        checkpoint = out / "checkpoint.pt"\n        start = 0\n        if args.resume:\n            ck = torch.load(checkpoint, map_location="cpu", weights_only=False)\n            if ck["contract"] != contract:\n                raise ValueError("resume contract differs: dataset/model/training settings must match")\n            lb.load_lora_state_dict(base, ck["lora"])\n            opt.load_state_dict(ck["optimizer"])\n            scaler.load_state_dict(ck["scaler"])\n            torch.set_rng_state(ck["rng"][rank]["cpu"])\n            if cuda:\n                torch.cuda.set_rng_state(ck["rng"][rank]["cuda"], device)\n            start = ck["step"]\n        elif out.exists() and any(out.iterdir()):\n            raise ValueError("output directory is not empty; use --resume or a new directory")\n        out.mkdir(parents=True, exist_ok=True)\n        def evaluate():\n            base.eval()\n            sums = {lang: [0.0, 0] for lang in languages}\n            with torch.no_grad():\n                for row in evaluated:\n                    batch = collate([row])\n                    with torch.autocast(device.type, dtype=dtype, enabled=cuda):\n                        value = base(**batch).loss.item()\n                    if not math.isfinite(value):\n                        raise ValueError("nonfinite validation loss")\n                    count = int((batch["labels"][:, 1:] != -100).sum())\n                    sums[row["language"]][0] += value * count\n                    sums[row["language"]][1] += count\n            base.train()\n            return {lang: total / n for lang, (total, n) in sums.items() if n}\n\n        if not args.resume and rank == 0:\n            baseline = evaluate()\n            (out / "baseline.json").write_text(json.dumps(baseline, indent=2), encoding="utf-8")\n            print(json.dumps({"step": 0, "validation_assistant_loss": baseline}), flush=True)\n        if world > 1:\n            dist.barrier()\n        sampler = DistributedSampler(encoded, num_replicas=world, rank=rank, seed=args.seed)\n        loader = DataLoader(encoded, batch_size=args.batch, sampler=sampler, collate_fn=collate,\n                            generator=torch.Generator().manual_seed(args.seed))\n        micro = start * args.accum\n        epoch, offset = divmod(micro, len(loader))\n        sampler.set_epoch(epoch)\n        iterator = iter(loader)\n        for _ in range(offset):\n            next(iterator)\n        model.train()\n        started_at = time.monotonic()\n        for step in range(start, args.steps):\n            opt.zero_grad(set_to_none=True)\n            for _ in range(args.accum):\n                try:\n                    batch = next(iterator)\n                except StopIteration:\n                    epoch += 1\n                    sampler.set_epoch(epoch)\n                    iterator = iter(loader)\n                    batch = next(iterator)\n                with torch.autocast(device.type, dtype=dtype, enabled=cuda):\n                    loss = model(**batch).loss / args.accum\n                if not torch.isfinite(loss):\n                    raise ValueError("nonfinite training loss")\n                scaler.scale(loss).backward()\n            scaler.unscale_(opt)\n            torch.nn.utils.clip_grad_norm_(params, 1.0, error_if_nonfinite=True)\n            scaler.step(opt)\n            scaler.update()\n            elapsed = time.monotonic() - started_at\n            stop = torch.tensor(int(rank == 0 and args.max_runtime_minutes > 0 and elapsed >= args.max_runtime_minutes * 60), device=device)\n            if world > 1:\n                dist.broadcast(stop, src=0)\n            stopping = bool(stop.item())\n            if rank == 0 and (step + 1) % args.log_every == 0:\n                print(json.dumps({"step": step + 1, "elapsed_seconds": elapsed,\n                    "seconds_per_step": elapsed / (step + 1 - start),\n                    "peak_gpu_gb": torch.cuda.max_memory_allocated(device) / 1e9 if cuda else 0}), flush=True)\n            if (step + 1) % args.checkpoint_every and step + 1 != args.steps and not stopping:\n                continue\n            rng = {"cpu": torch.get_rng_state(), "cuda": torch.cuda.get_rng_state(device) if cuda else None}\n            states = [None] * world\n            if world > 1:\n                dist.all_gather_object(states, rng)\n            else:\n                states[0] = rng\n            if rank == 0:\n                metrics = evaluate()\n                print(json.dumps({"step": step + 1, "validation_assistant_loss": metrics}))\n                payload = {"step": step + 1, "lora": lb.lora_state_dict(base), "optimizer": opt.state_dict(),\n                           "scaler": scaler.state_dict(), "rng": states, "contract": contract, "metrics": metrics}\n                torch.save(payload, out / "checkpoint.tmp")\n                os.replace(out / "checkpoint.tmp", checkpoint)\n                tensors = {}\n                for name, entry in payload["lora"].items():\n                    for key in ("A", "B"):\n                        tensors[lb.export_key(name) + "." + key] = entry[key].detach().cpu().float().contiguous()\n                # Immutable step exports avoid mismatched sidecar/weights after interruption.\n                prefix = out / f"adapter-step{step + 1}"\n                save_file(tensors, str(prefix) + ".safetensors")\n                meta = {"base": {"kind": "smoke" if args.smoke else "offline"}, "lora": {"r": args.rank, "alpha": args.alpha,\n                        "scaling": args.alpha / args.rank, "n_modules": len(payload["lora"])}, "languages": languages,\n                        "contract": contract, "validation_assistant_loss": metrics}\n                Path(str(prefix) + ".json").write_text(json.dumps(meta, indent=2), encoding="utf-8")\n                base.train()\n            if world > 1:\n                dist.barrier()\n            if stopping:\n                break\n    finally:\n        if world > 1 and dist.is_initialized():\n            dist.destroy_process_group()\n\n\nif __name__ == "__main__":\n    main()\n', 'forge/tool_data.py': '"""Validated multilingual tool trajectories; the first release trains English.\n\nJSONL rows: {language: "en", messages: [{role, content}, ...]}.\nRoles use the exact System/User/Assistant/Tool text protocol of cortex.Runner.\nSupply the serving system prompt as the first message in every trajectory.\n"""\nimport hashlib\nimport json\nimport unicodedata\nfrom collections import Counter\n\nENCODING_VERSION = "assistant-header-split-v2"\n\n\ndef load_trajectories(path, languages=("en",)):\n    rows = []\n    with open(path, encoding="utf-8") as source:\n        for number, line in enumerate(source, 1):\n            if not line.strip():\n                continue\n            row = json.loads(line)\n            lang = row.get("language")\n            if lang not in languages:\n                raise ValueError(f"{path}:{number}: language {lang!r} is not enabled")\n            messages = row.get("messages", [])\n            if len(messages) < 3 or messages[0].get("role") != "system" or messages[-1].get("role") != "assistant":\n                raise ValueError(f"{path}:{number}: expected system, conversation, final assistant answer")\n            previous = "system"\n            for i, message in enumerate(messages):\n                role, content = message.get("role"), message.get("content")\n                if not isinstance(content, str) or not content.strip() or "<|" in content:\n                    raise ValueError(f"{path}:{number}: empty content or embedded control token")\n                if i == 0:\n                    continue\n                allowed = {"system": ("user",), "user": ("assistant",),\n                           "assistant": ("user", "tool"), "tool": ("assistant",)}[previous]\n                if role not in allowed:\n                    raise ValueError(f"{path}:{number}: invalid role transition {previous} -> {role}")\n                prev_content = messages[i - 1]["content"].strip()\n                if previous == "assistant" and (role == "tool") != prev_content.startswith("CALL "):\n                    raise ValueError(f"{path}:{number}: each CALL requires a tool observation")\n                previous = role\n            if messages[-1]["content"].strip().startswith("CALL "):\n                raise ValueError(f"{path}:{number}: missing tool observation/final answer")\n            rows.append(row)\n    if not rows:\n        raise ValueError(f"{path}: empty dataset")\n    return rows\n\n\ndef task_key(row):\n    # Group paraphrases separately during curation; this catches exact prompt\n    # duplicates even if their answers, tool observations or language tags differ.\n    prompt = next(m["content"] for m in row["messages"] if m["role"] == "user")\n    prompt = " ".join(unicodedata.normalize("NFKC", prompt).casefold().split())\n    return hashlib.sha256(prompt.encode("utf-8")).hexdigest()\n\n\ndef validate_splits(train, validation):\n    overlap = {task_key(r) for r in train} & {task_key(r) for r in validation}\n    shared_ids = {r["task_id"] for r in train if r.get("task_id")} & {r["task_id"] for r in validation if r.get("task_id")}\n    if shared_ids:\n        raise ValueError(f"training/validation leakage: {len(shared_ids)} shared task IDs (including translations)")\n    if overlap:\n        raise ValueError(f"training/validation leakage: {len(overlap)} shared initial prompts")\n    return {"train": dict(Counter(r["language"] for r in train)),\n            "validation": dict(Counter(r["language"] for r in validation))}\n\n\ndef encode_trajectory(row, tokenizer, max_length):\n    ids, labels = [], []\n    for message in row["messages"]:\n        role = message["role"]\n        content = message["content"].strip() + "<|eot_id|>"\n        if role == "assistant":\n            # Go feeds the complete generation header before sampling. Encoding\n            # header+answer together merges its trailing space into the first\n            # answer token, creating a different prefix from live inference.\n            header = tokenizer.encode("Assistant: ", add_special_tokens=False)\n            answer = tokenizer.encode(content, add_special_tokens=False)\n            ids.extend(header + answer)\n            labels.extend([-100] * len(header) + answer)\n        else:\n            full = tokenizer.encode(role.capitalize() + ": " + content, add_special_tokens=False)\n            ids.extend(full)\n            labels.extend([-100] * len(full))\n    if len(ids) > max_length:\n        raise ValueError(f"trajectory has {len(ids)} tokens > {max_length}; curate it instead of truncating actions")\n    return {"input_ids": ids, "labels": labels, "language": row["language"]}\n', 'forge/bitnet_reference.py': '"""forge/bitnet_reference.py — ground-truth logits from the real\n`transformers` BitNet implementation, for cortex/bitnet_equivalence_test.go\nto check the Go engine (cortex/bitnet.go, cortex/bitnet_linear.go) against.\n\nTwo modes:\n\n  (default) loads the real 2.4B microsoft/bitnet-b1.58-2B-4T checkpoint\n    from --hf-dir with `AutoModelForCausalLM.from_pretrained(...,\n    torch_dtype=torch.float32, low_cpu_mem_usage=True)`, tokenizes 6\n    prompts (3 English, 3 Romanian), and writes\n    data/forge/bitnet-2b4t/logits_ref.json.\n\n  --tiny builds a small synthetic BitNet (2 layers, hidden 32, 4 heads /\n    2 kv heads, ffn 48, vocab 64) with ONLINE ternary quantization, runs\n    it on synthetic token-id prompts, and exports the ternary weights it\n    actually used through forge/import_bitnet.py\'s write_top/write_layer\n    — the SAME importer code path the real checkpoint goes through — to\n    forge/fixtures/bitnet_tiny.nxtf, with matching reference logits at\n    forge/fixtures/bitnet_tiny.json. This exercises the whole chain\n    (quantize -> pack -> NXTF -> Go load -> Go forward) without needing\n    the 2.4B checkpoint.\n\nIMPORTANT: `transformers`\' BitNet quantization ops (WeightQuant.forward,\nActQuant.forward, BitLinear.activation_quant/post_quant_process, and the\ntop-level unpack_weights used by AutoBitLinear\'s state-dict load hook) are\nall decorated with @torch.compile. On this machine that tries to invoke\nMSVC (`cl.exe`) via the inductor backend and fails with "Compiler: cl is\nnot found". Disabling TorchDynamo makes @torch.compile a no-op (falls back\nto eager — bit-exact, just slower), so the env var below is set before\n`torch` is imported anywhere in the process.\n\n    python forge/bitnet_reference.py --tiny\n    python forge/bitnet_reference.py --hf-dir data/pretrained/bitnet-b1.58-2B-4T \\\n        --out-dir data/forge/bitnet-2b4t\n"""\n\nfrom __future__ import annotations\n\nimport os\n\nos.environ.setdefault("TORCHDYNAMO_DISABLE", "1")\n\nimport argparse  # noqa: E402\nimport json  # noqa: E402\nimport sys  # noqa: E402\nimport time  # noqa: E402\n\nimport numpy as np  # noqa: E402\nimport torch  # noqa: E402\n\nsys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))\nfrom import_bitnet import NXTFWriter, write_layer, write_top  # noqa: E402\n\nEN_PROMPTS = [\n    "The capital of France is",\n    "In machine learning, a neural network is",\n    "She opened the door and",\n]\nRO_PROMPTS = [\n    "Capitala Romaniei este",\n    "Inteligenta artificiala este un domeniu care",\n    "A deschis usa si",\n]\n\nGREEDY_TOKENS_REAL = 8\nGREEDY_TOKENS_TINY = 5\n\n\ndef greedy_generate(model, ids: list[int], max_new: int, eos_token_id: int | None) -> list[int]:\n    """Manual no-KV-cache greedy loop — matches\n    BitNetModel.GenerateGreedy\'s "recompute Forward over the whole\n    sequence so far" approach exactly, so both sides do the identical\n    sequence of forward passes."""\n    cur = torch.tensor([ids], dtype=torch.long)\n    generated: list[int] = []\n    for _ in range(max_new):\n        with torch.no_grad():\n            out = model(cur)\n        nxt = int(out.logits[0, -1].argmax())\n        generated.append(nxt)\n        cur = torch.cat([cur, torch.tensor([[nxt]], dtype=torch.long)], dim=1)\n        if eos_token_id is not None and nxt == eos_token_id:\n            break\n    return generated\n\n\ndef run_prompt(model, ids: list[int], max_new: int, eos_token_id: int | None) -> dict:\n    input_ids = torch.tensor([ids], dtype=torch.long)\n    with torch.no_grad():\n        out = model(input_ids)\n    logits = out.logits[0].to(torch.float32)  # (T, V)\n    return {\n        "ids": ids,\n        "logits_last": [round(float(x), 6) for x in logits[-1].tolist()],\n        "argmax": [int(v) for v in logits.argmax(dim=-1).tolist()],\n        "greedy_continuation": greedy_generate(model, ids, max_new, eos_token_id),\n    }\n\n\n# ---------------------------------------------------------------------------\n# Real 2.4B checkpoint\n# ---------------------------------------------------------------------------\n\ndef _fix_unmaterialized_offline_weights(model, hf_dir: str) -> int:\n    """Works around a loading bug observed with the installed transformers\n    5.3.0 + this checkpoint: `AutoModelForCausalLM.from_pretrained(...,\n    torch_dtype=torch.float32, low_cpu_mem_usage=True)` completes without\n    error and produces a model whose AutoBitLinear.weight_scale buffers are\n    correctly loaded, but whose `.weight` tensors are LEFT PACKED — dtype\n    torch.uint8 instead of the unpacked ternary float the offline\n    (non-online-quant) AutoBitLinear.forward requires. Forward then raises:\n\n        RuntimeError: expected m1 and m2 to have the same dtype,\n        but got: float != unsigned char\n        (transformers/integrations/bitnet.py:309, AutoBitLinear.forward,\n         output = F.linear(input, weight, self.bias))\n\n    AutoBitLinear.load_hook (bitnet.py:288-297) is supposed to unpack the\n    checkpoint\'s raw uint8 tensor via `unpack_weights(..., dtype=self.\n    weight.dtype)` before torch\'s normal state-dict copy runs, but\n    apparently doesn\'t fire (or fires with the wrong target dtype) on this\n    from_pretrained code path — root cause not fully isolated, but the\n    symptom is 100% reproducible: after loading, every offline AutoBitLinear\n    module\'s `.weight` is still torch.uint8 (confirmed via\n    `mod.weight.dtype` on `model.model.layers[0].self_attn.q_proj` right\n    after `from_pretrained` returns).\n\n    Rather than silently reporting a failure or downloading the separate\n    bf16 (non-quantized) repo this task allows as a fallback, this re-reads\n    each affected module\'s raw packed tensor straight from the SAME\n    already-local model.safetensors and unpacks it with\n    forge/import_bitnet.py\'s hf_unpack_packed — the exact function this\n    file\'s own NXTF importer uses and that cortex/bitnet_equivalence_test.go\n    already validates end-to-end via the tiny fixture — then assigns the\n    result as the module\'s new weight Parameter. No network access, no\n    extra files. Returns the number of modules fixed (210 for the real\n    checkpoint: 30 layers x 7 BitLinear projections)."""\n    from safetensors import safe_open\n    from transformers.integrations.bitnet import AutoBitLinear\n\n    fixed = 0\n    st_path = os.path.join(hf_dir, "model.safetensors")\n    with safe_open(st_path, framework="pt", device="cpu") as f:\n        for name, mod in model.named_modules():\n            if isinstance(mod, AutoBitLinear) and not mod.online_quant and mod.weight.dtype == torch.uint8:\n                packed = f.get_tensor(name + ".weight").numpy()\n                from import_bitnet import hf_unpack_packed  # local import: keeps the fix self-contained\n                ternary = hf_unpack_packed(packed, mod.out_features)\n                if ternary.shape != (mod.out_features, mod.in_features):\n                    raise ValueError(f"{name}: unpacked shape {ternary.shape}, want {(mod.out_features, mod.in_features)}")\n                mod.weight = torch.nn.Parameter(torch.from_numpy(ternary.astype(np.float32)), requires_grad=False)\n                fixed += 1\n    return fixed\n\n\ndef run_real(hf_dir: str, out_dir: str) -> None:\n    from transformers import AutoModelForCausalLM, AutoTokenizer\n\n    t0 = time.time()\n    tokenizer = AutoTokenizer.from_pretrained(hf_dir)\n    try:\n        model = AutoModelForCausalLM.from_pretrained(\n            hf_dir, torch_dtype=torch.float32, low_cpu_mem_usage=True,\n        )\n    except Exception as e:\n        print(f"[bitnet_reference] FAILED to load the offline-packed checkpoint from {hf_dir!r}: {type(e).__name__}: {e}")\n        print("[bitnet_reference] Not falling back to any other repo / downloading anything else automatically.")\n        raise\n    n_fixed = _fix_unmaterialized_offline_weights(model, hf_dir)\n    print(f"[bitnet_reference] worked around from_pretrained\'s unmaterialized-weight bug: re-unpacked {n_fixed} AutoBitLinear modules directly from {hf_dir}/model.safetensors")\n    model.eval()\n    print(f"[bitnet_reference] loaded {hf_dir} in {time.time() - t0:.1f}s")\n\n    prompts_text = EN_PROMPTS + RO_PROMPTS\n    results = []\n    for text in prompts_text:\n        ids = tokenizer(text)["input_ids"]\n        t1 = time.time()\n        r = run_prompt(model, ids, GREEDY_TOKENS_REAL, model.config.eos_token_id)\n        r["text"] = text\n        results.append(r)\n        print(f"[bitnet_reference] {text!r}: {len(ids)} ids, {time.time() - t1:.1f}s")\n\n    os.makedirs(out_dir, exist_ok=True)\n    out_path = os.path.join(out_dir, "logits_ref.json")\n    with open(out_path, "w", encoding="utf-8") as f:\n        json.dump({"hf_dir": hf_dir, "prompts": results}, f, indent=2)\n    print(f"[bitnet_reference] wrote {out_path}")\n\n\n# ---------------------------------------------------------------------------\n# Tiny synthetic model (online quantization) — exercises the whole chain\n# without the 2.4B checkpoint.\n# ---------------------------------------------------------------------------\n\ndef run_tiny(out_dir: str) -> None:\n    from transformers.integrations.bitnet import AutoBitLinear, replace_with_bitnet_linear\n    from transformers.models.bitnet.configuration_bitnet import BitNetConfig as HFBitNetConfig\n    from transformers.models.bitnet.modeling_bitnet import BitNetForCausalLM\n\n    torch.manual_seed(1234)\n    hidden, ffn, num_layers, heads, kv_heads, vocab = 32, 48, 2, 4, 2, 64\n    # Sentinel ids clearly outside the tiny vocab so greedy generation never\n    # early-stops on EOS on either the Python or the Go side — keeps the\n    # continuation length fixed and the comparison unambiguous.\n    bos_id, eos_id = 9998, 9999\n\n    hf_cfg = HFBitNetConfig(\n        vocab_size=vocab, hidden_size=hidden, intermediate_size=ffn,\n        num_hidden_layers=num_layers, num_attention_heads=heads, num_key_value_heads=kv_heads,\n        max_position_embeddings=64, rope_theta=10000.0, rms_norm_eps=1e-5,\n        tie_word_embeddings=True, attention_bias=False,\n        bos_token_id=bos_id, eos_token_id=eos_id,\n    )\n    model = BitNetForCausalLM(hf_cfg)\n\n    class _QC:\n        linear_class = "autobitlinear"\n        quantization_mode = "online"\n        use_rms_norm = False\n        rms_norm_eps = 1e-5\n\n    replace_with_bitnet_linear(model, modules_to_not_convert=["lm_head"], quantization_config=_QC())\n\n    # replace_with_bitnet_linear builds the new AutoBitLinear modules under\n    # `with torch.device("meta")` (it\'s designed for the from_pretrained\n    # loading path, where a state-dict load materializes them afterward).\n    # We aren\'t loading a checkpoint, so `.weight` is left as an empty meta\n    # tensor here. Left alone, F.linear(real_input, meta_weight) does NOT\n    # raise — this torch version silently returns a real all-zero tensor —\n    # so a forward pass "succeeds" but is meaningless. Materialize each\n    # converted module\'s weight with real random values before running\n    # anything.\n    for mod in model.modules():\n        if isinstance(mod, AutoBitLinear):\n            w = torch.empty(mod.weight.shape, dtype=torch.float32)\n            w.normal_(mean=0.0, std=1.0)\n            mod.weight = torch.nn.Parameter(w, requires_grad=False)\n    model.eval()\n\n    cfg = {\n        "vocab_size": vocab, "embed_dim": hidden, "num_layers": num_layers,\n        "num_heads": heads, "num_kv_heads": kv_heads, "ffn_dim": ffn,\n        "max_seq_len": 64, "rope_theta": 10000.0, "rms_norm_eps": 1e-5,\n        "bos_token_id": bos_id, "eos_token_id": eos_id,\n    }\n\n    def ternary_and_scale(mod: "AutoBitLinear") -> tuple[np.ndarray, float]:\n        """Reproduces WeightQuant.forward\'s formula exactly (the ternary\n        codes + dequant scale AutoBitLinear.forward actually used for this\n        module during the forward passes below), read straight from\n        mod.weight rather than through the torch.compile-wrapped autograd\n        Function — the formula is deterministic, so this is bit-exact."""\n        w = mod.weight.detach().float()\n        mean_abs = w.abs().mean().clamp(min=1e-5)\n        scale_pt = 1.0 / mean_abs\n        ternary = (w * scale_pt).round().clamp(-1, 1)\n        return ternary.to(torch.int8).numpy(), float(mean_abs.item())\n\n    os.makedirs(out_dir, exist_ok=True)\n    nxtf_path = os.path.join(out_dir, "bitnet_tiny.nxtf")\n    writer = NXTFWriter(nxtf_path, "bitnet", cfg)\n    embed = model.model.embed_tokens.weight.detach().float().numpy()\n    final_norm = model.model.norm.weight.detach().float().numpy()\n    write_top(writer, embed, final_norm)\n\n    for li in range(num_layers):\n        dl = model.model.layers[li]\n        layer = {\n            "attn_norm": dl.input_layernorm.weight.detach().float().numpy(),\n            "ffn_norm": dl.post_attention_layernorm.weight.detach().float().numpy(),\n            "attn_sub_norm": dl.self_attn.attn_sub_norm.weight.detach().float().numpy(),\n            "ffn_sub_norm": dl.mlp.ffn_sub_norm.weight.detach().float().numpy(),\n            "q": ternary_and_scale(dl.self_attn.q_proj),\n            "k": ternary_and_scale(dl.self_attn.k_proj),\n            "v": ternary_and_scale(dl.self_attn.v_proj),\n            "o": ternary_and_scale(dl.self_attn.o_proj),\n            "gate": ternary_and_scale(dl.mlp.gate_proj),\n            "up": ternary_and_scale(dl.mlp.up_proj),\n            "down": ternary_and_scale(dl.mlp.down_proj),\n        }\n        write_layer(writer, li, layer)\n    writer.finalize()\n    print(f"[bitnet_reference] wrote {nxtf_path}")\n\n    rng = np.random.default_rng(99)\n    prompts = [\n        [int(x) for x in rng.integers(2, vocab, size=6)],\n        [int(x) for x in rng.integers(2, vocab, size=4)],\n        [int(x) for x in rng.integers(2, vocab, size=8)],\n    ]\n    results = []\n    for ids in prompts:\n        r = run_prompt(model, ids, GREEDY_TOKENS_TINY, eos_id)\n        r["text"] = None\n        results.append(r)\n\n    json_path = os.path.join(out_dir, "bitnet_tiny.json")\n    with open(json_path, "w", encoding="utf-8") as f:\n        json.dump({"config": cfg, "prompts": results}, f, indent=2)\n    print(f"[bitnet_reference] wrote {json_path}")\n\n\ndef main() -> None:\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--tiny", action="store_true", help="build+run the tiny synthetic model instead of the real checkpoint")\n    ap.add_argument("--hf-dir", default="data/pretrained/bitnet-b1.58-2B-4T")\n    ap.add_argument("--out-dir", default="data/forge/bitnet-2b4t")\n    ap.add_argument("--tiny-out-dir", default="forge/fixtures")\n    args = ap.parse_args()\n    if args.tiny:\n        run_tiny(args.tiny_out_dir)\n    else:\n        run_real(args.hf_dir, args.out_dir)\n\n\nif __name__ == "__main__":\n    main()\n', 'forge/import_bitnet.py': '"""import_bitnet.py — convert the packed HF BitNet b1.58 checkpoint (or any\nin-memory BitNet-shaped state dict) into NXTF v3 (`bitnet.nxtf`), the format\n`cortex.LoadBitNetModel` reads.\n\nTwo entry points share the same tensor-writing code (the "importer code\npath" the task asks both the real model and the `--tiny` synthetic fixture\nto go through):\n\n    write_top(writer, embed, final_norm)      — embedding + final RMSNorm\n    write_layer(writer, layer_idx, layer)      — one decoder layer\n\n`import_from_hf_dir` builds each layer\'s arrays straight from the packed\nsafetensors (via `safe_open`, so only one tensor is materialized at a time —\nthe whole 2.4B-param checkpoint never needs to live in RAM at once) and\nfeeds them through those two functions. `forge/bitnet_reference.py --tiny`\nbuilds a random synthetic BitNet\'s ternary weights the same shape and feeds\nthem through the identical `write_top`/`write_layer` pair.\n\nWeight-scale convention (verified against the installed transformers\n5.3.0, transformers/integrations/bitnet.py, AutoBitLinear.forward,\n`quantization_mode="offline"` path — the mode this checkpoint\'s\nconfig.json declares):\n\n    if self.online_quant:\n        weight = WeightQuant.apply(self.weight)\n    else:\n        weight = self.weight          # already unpacked to {-1,0,1} by load_hook\n    input = ActQuant.apply(input)\n    output = F.linear(input, weight, self.bias)\n    if not self.online_quant:\n        output = output * self.weight_scale\n\n`self.weight` after `AutoBitLinear.load_hook` has already been unpacked\n(via `unpack_weights`) to plain ternary values in {-1, 0, 1} — never\nscaled. The *only* place weight_scale is applied is as a scalar multiply\non the matmul output, which is mathematically identical to first\ndequantizing the weight matrix as `W_real = weight_scale * W_ternary` and\nthen doing a normal matmul. So the convention is MULTIPLY:\n\n    W ≈ scale · T\n\nwhich is exactly what the NXTF v3 header\'s per-tensor "scale" field means,\nand exactly what `BitLinear.Scale` is used for on the Go side.\n\nPacking layout (ternary tensors): mirrors cortex/ternary.go\'s\nPackTernaryTile bit-for-bit — see pack_ternary_tile() below. Row-major\nover Out, ceil(In/16) tiles per row, each tile a little-endian uint32.\n\nThe NXTF v3 container itself (magic/header/blob streaming — the `NXTFWriter`\nclass) lives in forge/nxtf3.py, shared with forge/multimodal/export_tower.py\n(arch "siglip2_vision"); this module still owns everything "bitnet"-shaped\n(write_top/write_layer, ternary packing, HF-checkpoint import) and always\ncalls NXTFWriter with arch="bitnet", so its output is byte-for-byte\nunchanged from before that refactor.\n\nHF\'s on-disk packing (unrelated 2-bit format, decoded here via\nhf_unpack_packed) is different: transformers packs 4 output-rows per\nuint8 byte (see transformers/integrations/bitnet.py: pack_weights /\nunpack_weights, VALUES_PER_ITEM=4). We unpack THAT format fully to a\nplain {-1,0,1} int8 matrix first, then re-pack it into our own\nPackTernaryTile layout for the Go engine.\n\n    python forge/import_bitnet.py --hf-dir data/pretrained/bitnet-b1.58-2B-4T \\\n        --out data/forge/bitnet-2b4t/bitnet.nxtf\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport os\nfrom typing import Any\n\nimport numpy as np\n\nfrom nxtf3 import NXTFWriter, f32_bytes as _f32_bytes\n\n# ---------------------------------------------------------------------------\n# Ternary packing — mirrors cortex/ternary.go: PackTernaryTile exactly.\n# ---------------------------------------------------------------------------\n#\n# Go (cortex/ternary.go):\n#   R byte = signLo (bit i set => weight i, i in [0,8) is negative)\n#   G byte = maskLo (bit i set => weight i, i in [0,8) is non-zero)\n#   B byte = signHi (bit i set => weight 8+i is negative)\n#   A byte = maskHi (bit i set => weight 8+i is non-zero)\n#   tile uint32 = signLo | maskLo<<8 | signHi<<16 | maskHi<<24\n#   stored little-endian on disk (binary.LittleEndian.PutUint32).\n\n_BIT = (1 << np.arange(8, dtype=np.uint32))  # [1,2,4,...,128]\n\n\ndef pack_ternary_tile(weights16: list[int]) -> int:\n    """Scalar reference implementation — pack exactly 16 ternary weights\n    ({-1,0,1}) into one uint32 tile, bit-for-bit identical to Go\'s\n    PackTernaryTile. Used to sanity-check the vectorized path and to build\n    forge/fixtures/ternary_pack_fixture.json."""\n    assert len(weights16) == 16\n    sign_lo = mask_lo = sign_hi = mask_hi = 0\n    for i in range(8):\n        w = weights16[i]\n        if w != 0:\n            mask_lo |= 1 << i\n            if w < 0:\n                sign_lo |= 1 << i\n    for i in range(8):\n        w = weights16[8 + i]\n        if w != 0:\n            mask_hi |= 1 << i\n            if w < 0:\n                sign_hi |= 1 << i\n    return (sign_lo | (mask_lo << 8) | (sign_hi << 16) | (mask_hi << 24)) & 0xFFFFFFFF\n\n\ndef pack_ternary_matrix(mat: np.ndarray) -> bytes:\n    """Vectorized version of pack_ternary_tile applied row-major over a\n    (out_features, in_features) int8 matrix with values in {-1,0,1}.\n    Returns raw bytes: out_features * ceil(in_features/16) little-endian\n    uint32 tiles, row-major over `out`, tiles left-to-right along `in`\n    within a row (rows beyond padding are zero-filled, matching Go\'s\n    zero-initialized Tiles for positions >= InputSize).\n    """\n    if mat.dtype != np.int8:\n        mat = mat.astype(np.int8)\n    out_f, in_f = mat.shape\n    tiles_per_row = (in_f + 15) // 16\n    padded_len = tiles_per_row * 16\n    if padded_len != in_f:\n        padded = np.zeros((out_f, padded_len), dtype=np.int8)\n        padded[:, :in_f] = mat\n    else:\n        padded = mat\n    tiled = padded.reshape(out_f, tiles_per_row, 16)\n    lo = tiled[:, :, :8]\n    hi = tiled[:, :, 8:16]\n\n    mask_lo = (lo != 0)\n    sign_lo = (lo < 0)\n    mask_hi = (hi != 0)\n    sign_hi = (hi < 0)\n\n    sign_lo_byte = (sign_lo.astype(np.uint32) * _BIT).sum(axis=-1).astype(np.uint32)\n    mask_lo_byte = (mask_lo.astype(np.uint32) * _BIT).sum(axis=-1).astype(np.uint32)\n    sign_hi_byte = (sign_hi.astype(np.uint32) * _BIT).sum(axis=-1).astype(np.uint32)\n    mask_hi_byte = (mask_hi.astype(np.uint32) * _BIT).sum(axis=-1).astype(np.uint32)\n\n    tile_u32 = (\n        sign_lo_byte\n        | (mask_lo_byte << 8)\n        | (sign_hi_byte << 16)\n        | (mask_hi_byte << 24)\n    )  # (out_f, tiles_per_row) uint32, row-major over out already via C order\n\n    return np.ascontiguousarray(tile_u32, dtype="<u4").tobytes()\n\n\n# ---------------------------------------------------------------------------\n# HF packed-uint8 -> plain ternary int8, mirrors\n# transformers/integrations/bitnet.py: unpack_weights exactly.\n# ---------------------------------------------------------------------------\n\nVALUES_PER_ITEM = 4  # HF packs 4 ternary values (2 bits each) per uint8\n\n\ndef hf_unpack_packed(packed: np.ndarray, out_features: int) -> np.ndarray:\n    """packed: (row_dim, in_features) uint8, row_dim = ceil(out_features/4).\n    Returns (out_features, in_features) int8 with values in {-1,0,1}."""\n    row_dim = packed.shape[0]\n    full = np.zeros((row_dim * VALUES_PER_ITEM,) + packed.shape[1:], dtype=np.int8)\n    for i in range(VALUES_PER_ITEM):\n        codes = ((packed >> (2 * i)) & 3).astype(np.int8) - 1\n        full[i * row_dim:(i + 1) * row_dim, ...] = codes\n    return full[:out_features, ...]\n\n\ndef write_top(writer: NXTFWriter, embed: np.ndarray, final_norm: np.ndarray) -> None:\n    """Shared importer step: embedding table + final RMSNorm weight.\n    Called identically by the real-checkpoint importer and the --tiny\n    synthetic-fixture exporter."""\n    writer.add("embed", "f32", list(embed.shape), _f32_bytes(embed))\n    writer.add("final_norm", "f32", list(final_norm.shape), _f32_bytes(final_norm))\n\n\n_LAYER_NORM_KEYS = ("attn_norm", "ffn_norm", "attn_sub_norm", "ffn_sub_norm")\n_LAYER_LINEAR_TAGS = ("q", "k", "v", "o", "gate", "up", "down")\n\n\ndef write_layer(writer: NXTFWriter, layer_idx: int, layer: dict[str, Any]) -> None:\n    """Shared importer step: one decoder layer\'s norms + 7 BitLinear\n    tensors. `layer[norm_key]` is a 1-D float array; `layer[tag]` is a\n    (ternary_int8_matrix, scale) pair. Called identically by the\n    real-checkpoint importer and the --tiny synthetic-fixture exporter."""\n    prefix = f"layers.{layer_idx}."\n    for key in _LAYER_NORM_KEYS:\n        arr = layer[key]\n        writer.add(prefix + key, "f32", list(arr.shape), _f32_bytes(arr))\n    for tag in _LAYER_LINEAR_TAGS:\n        ternary, scale = layer[tag]\n        writer.add(prefix + tag, "ternary", list(ternary.shape), pack_ternary_matrix(ternary), scale=scale)\n\n\ndef export_bitnet_nxtf(out_path: str, config: dict[str, Any], embed: np.ndarray,\n                        final_norm: np.ndarray, layers: list[dict[str, Any]]) -> None:\n    """Convenience wrapper: write a complete model in one call (used by the\n    --tiny fixture generator, which holds everything in memory already)."""\n    w = NXTFWriter(out_path, "bitnet", config)\n    write_top(w, embed, final_norm)\n    for i, layer in enumerate(layers):\n        write_layer(w, i, layer)\n    w.finalize()\n\n\n# ---------------------------------------------------------------------------\n# Real-checkpoint import\n# ---------------------------------------------------------------------------\n\ndef bitnet_config_from_hf(hf_cfg: dict[str, Any]) -> dict[str, Any]:\n    return {\n        "vocab_size": hf_cfg["vocab_size"],\n        "embed_dim": hf_cfg["hidden_size"],\n        "num_layers": hf_cfg["num_hidden_layers"],\n        "num_heads": hf_cfg["num_attention_heads"],\n        "num_kv_heads": hf_cfg["num_key_value_heads"],\n        "ffn_dim": hf_cfg["intermediate_size"],\n        "max_seq_len": hf_cfg["max_position_embeddings"],\n        "rope_theta": float(hf_cfg["rope_theta"]),\n        "rms_norm_eps": float(hf_cfg["rms_norm_eps"]),\n        "bos_token_id": hf_cfg["bos_token_id"],\n        "eos_token_id": hf_cfg["eos_token_id"],\n    }\n\n\ndef import_from_hf_dir(hf_dir: str, out_path: str) -> None:\n    import torch\n    from safetensors import safe_open\n\n    with open(os.path.join(hf_dir, "config.json")) as f:\n        hf_cfg = json.load(f)\n    cfg = bitnet_config_from_hf(hf_cfg)\n    head_dim = cfg["embed_dim"] // cfg["num_heads"]\n\n    specs = [\n        ("q", "self_attn.q_proj", cfg["num_heads"] * head_dim, cfg["embed_dim"]),\n        ("k", "self_attn.k_proj", cfg["num_kv_heads"] * head_dim, cfg["embed_dim"]),\n        ("v", "self_attn.v_proj", cfg["num_kv_heads"] * head_dim, cfg["embed_dim"]),\n        ("o", "self_attn.o_proj", cfg["embed_dim"], cfg["num_heads"] * head_dim),\n        ("gate", "mlp.gate_proj", cfg["ffn_dim"], cfg["embed_dim"]),\n        ("up", "mlp.up_proj", cfg["ffn_dim"], cfg["embed_dim"]),\n        ("down", "mlp.down_proj", cfg["embed_dim"], cfg["ffn_dim"]),\n    ]\n\n    st_path = os.path.join(hf_dir, "model.safetensors")\n    writer = NXTFWriter(out_path, "bitnet", cfg)\n\n    with safe_open(st_path, framework="pt", device="cpu") as f:\n        def f32(name: str) -> np.ndarray:\n            t = f.get_tensor(name).to(torch.float32).contiguous()\n            return t.numpy()\n\n        write_top(writer, f32("model.embed_tokens.weight"), f32("model.norm.weight"))\n\n        for li in range(cfg["num_layers"]):\n            p = f"model.layers.{li}."\n            layer = {\n                "attn_norm": f32(p + "input_layernorm.weight"),\n                "ffn_norm": f32(p + "post_attention_layernorm.weight"),\n                "attn_sub_norm": f32(p + "self_attn.attn_sub_norm.weight"),\n                "ffn_sub_norm": f32(p + "mlp.ffn_sub_norm.weight"),\n            }\n            for tag, mod, out_f, in_f in specs:\n                packed = f.get_tensor(p + mod + ".weight").numpy()  # uint8\n                ternary = hf_unpack_packed(packed, out_f)\n                if ternary.shape != (out_f, in_f):\n                    raise ValueError(f"{p}{mod}: unpacked shape {ternary.shape}, want {(out_f, in_f)}")\n                scale = float(f.get_tensor(p + mod + ".weight_scale").to(torch.float32).item())\n                layer[tag] = (ternary, scale)\n            write_layer(writer, li, layer)\n            del layer\n\n    writer.finalize()\n    print(f"[import_bitnet] wrote {out_path}")\n\n\ndef main() -> None:\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--hf-dir", default="data/pretrained/bitnet-b1.58-2B-4T")\n    ap.add_argument("--out", default="data/forge/bitnet-2b4t/bitnet.nxtf")\n    args = ap.parse_args()\n    import_from_hf_dir(args.hf_dir, args.out)\n\n\nif __name__ == "__main__":\n    main()\n', 'forge/nxtf3.py': '"""forge/nxtf3.py -- shared NXTF v3 (`NXTF3BIN`) writer, factored out of\nforge/import_bitnet.py so a second exporter (forge/multimodal/export_tower.py,\nwhich writes a `siglip2_vision` arch rather than `bitnet`) can produce the\nsame container format without duplicating the streaming-writer code or\nimport_bitnet.py\'s `arch` string.\n\nFormat (little-endian throughout):\n\n    magic   8 bytes  "NXTF3BIN"\n    hdrLen  uint32   length of the JSON header that follows\n    header  JSON     {"version": 3, "arch": <str>, "config": {...},\n                      "tensors": [{"name","kind","shape","offset","bytes",\n                                   "scale"?}, ...]}\n    blobs            each tensor\'s raw bytes, back-to-back; a tensor\'s\n                      "offset"/"bytes" fields are relative to the start of\n                      the blob region (immediately after the header).\n\n`arch` and `config` are caller-supplied and opaque to this module -- it only\nstreams tensor blobs to a temp file (so it never holds more than one\ntensor\'s raw bytes in memory at once) and concatenates\nmagic+header+blobs into the final file on `finalize()`. import_bitnet.py\'s\nown `write_top`/`write_layer`/ternary-packing helpers stay there (they are\n`arch: "bitnet"`-specific); this module only owns the container.\n\n`import_bitnet.py` imports `NXTFWriter` from here and calls it with\n`arch="bitnet"`, so its output is byte-for-byte unchanged from before this\nmodule existed.\n"""\n\nfrom __future__ import annotations\n\nimport json\nimport os\nimport shutil\nimport struct\nfrom typing import Any\n\nMAGIC = b"NXTF3BIN"\n\n\nclass NXTFWriter:\n    """Streams tensor blobs to a temp file, then concatenates\n    magic+header+blobs into the final file on finalize(). `arch` is written\n    verbatim into the header\'s "arch" field (e.g. "bitnet", "siglip2_vision")\n    -- the Go-side loader for that arch rejects any other value."""\n\n    def __init__(self, out_path: str, arch: str, config: dict[str, Any]):\n        self.out_path = out_path\n        self.arch = arch\n        self.config = config\n        self.tensors: list[dict[str, Any]] = []\n        os.makedirs(os.path.dirname(out_path) or ".", exist_ok=True)\n        self._tmp_path = out_path + ".blob.tmp"\n        self._tmp = open(self._tmp_path, "wb")\n        self._offset = 0\n\n    def add(self, name: str, kind: str, shape: list[int], data: bytes, scale: float | None = None) -> None:\n        entry: dict[str, Any] = {\n            "name": name,\n            "kind": kind,\n            "shape": list(shape),\n            "offset": self._offset,\n            "bytes": len(data),\n        }\n        if scale is not None:\n            entry["scale"] = float(scale)\n        self.tensors.append(entry)\n        self._tmp.write(data)\n        self._offset += len(data)\n\n    def finalize(self) -> None:\n        self._tmp.close()\n        header = {\n            "version": 3,\n            "arch": self.arch,\n            "config": self.config,\n            "tensors": self.tensors,\n        }\n        hdr_bytes = json.dumps(header).encode("utf-8")\n        with open(self.out_path, "wb") as out:\n            out.write(MAGIC)\n            out.write(struct.pack("<I", len(hdr_bytes)))\n            out.write(hdr_bytes)\n            with open(self._tmp_path, "rb") as tmp:\n                shutil.copyfileobj(tmp, out, length=4 * 1024 * 1024)\n        os.remove(self._tmp_path)\n\n\ndef f32_bytes(arr) -> bytes:\n    """np.ndarray (any shape/dtype) -> contiguous little-endian float32\n    bytes. Shared by every NXTF v3 writer that stores plain float tensors."""\n    import numpy as np\n\n    return np.ascontiguousarray(arr, dtype="<f4").tobytes()\n', 'forge/multimodal/mm_model.py': '"""forge/multimodal/mm_model.py -- BitNetVLM: frozen BitNet LLM + a trainable\nVisionAdapter (see vision_adapter.py), spliced together the LLaVA way.\n\nChat template (task spec, matching cortex.Llama3ChatPrompt\'s semantics in\ncortex/tokenizer_llama3.go and tokenizer_config.json\'s chat_template -- see\nthat file\'s own docstring):\n\n    "User: {content}<|eot_id|>Assistant: {answer}<|eot_id|>"\n\nFor a multimodal turn, {content} is the user\'s text with an IMAGE_TOKEN\n("<image>") marker somewhere inside it (data.py\'s samples always include\none). BitNetVLM:\n\n  1. splits {content} into the text before/after the marker,\n  2. tokenizes "User: " + before, and after + "<|eot_id|>Assistant: ",\n     each with the real tokenizer (add_special_tokens=False -- the template\n     above never adds a bos_token, matching Llama3ChatPrompt, which does not\n     either; see forge/llama3_tokenizer_reference.py\'s `chat` subcommand for\n     the validated one-shot-tokenize-the-rendered-string reference this\n     mirrors as closely as splicing an image allows),\n  3. looks up those ids\' embeddings via `model.get_input_embeddings()`,\n  4. runs the projected image tokens through the SAME embedding table\'s\n     hidden size (VisionAdapter already projects to llm_hidden) and\n     concatenates: [before_emb, image_emb, after_emb, answer_emb],\n  5. builds labels = [-100]*(everything up to and including the prompt) +\n     answer_ids, so the LM loss (out.loss from a standard HF CausalLM\n     forward with `labels=`) is computed on the answer tokens only.\n\nSplitting text around the image marker and tokenizing the pieces separately\n(step 2) can in principle land on slightly different BPE merges at the\nsplit boundary than tokenizing the whole string in one call would -- this\nis an accepted limitation shared by every LLaVA-style implementation that\nsplices image embeddings into a text token stream (there is no way to tokenize\n"through" an embedded image), not something specific to this code.\n\nThe LLM\'s parameters are frozen (`requires_grad=False`) but its forward is\nNOT wrapped in `torch.no_grad()`: gradients must still flow back through it\ninto the image embeddings so the projector (which sits before the LLM in the\ngraph) receives a gradient. This is the opposite of VisionAdapter\'s own\nfrozen tower, which IS wrapped in `torch.no_grad()` there, because nothing\ntrainable sits upstream of it.\n\nModality-generic since P4 "ears" (audio_adapter.py): BitNetVLM\'s splicing\nlogic never actually depended on the adapter being a VisionAdapter -- only\non it being callable as `adapter(features)` and returning something that\nindexes as `embeds[i]` per sample. `__init__` takes a `placeholder` argument\n(default IMAGE_PLACEHOLDER) so `_split_prompt` can look for AUDIO_PLACEHOLDER\n("<audio>") instead, and `encode(features)` calls `self.adapter(features)`\n(`self.adapter is self.vision_adapter`, kept as a name-stable attribute/\nparam for every eyes caller). train_stage1_audio.py builds a BitNetVLM with\nan audio_adapter.AudioAdapter in place of VisionAdapter and\nplaceholder=AUDIO_PLACEHOLDER; train_stage1.py/train_stage2.py/tests are\nuntouched and keep using `vision_adapter=`/`encode_images`/`generate_caption`\nexactly as before.\n"""\n\nfrom __future__ import annotations\n\nimport os\n\n# See forge/bitnet_reference.py\'s module docstring / train_stage1.py\'s own\n# copy of this line: must be set before torch is imported anywhere in the\n# process, so it is set here too in case this module is ever imported\n# before train_stage1.py sets it.\nos.environ.setdefault("TORCHDYNAMO_DISABLE", "1")\n\nimport sys  # noqa: E402\n\nimport torch  # noqa: E402\nimport torch.nn as nn\n\nIMAGE_PLACEHOLDER = "<image>"\nAUDIO_PLACEHOLDER = "<audio>"  # audio_adapter.AudioAdapter\'s marker (ears, P4) -- symmetry with IMAGE_PLACEHOLDER\n\n\ndef pad_and_stack(samples: list):\n    """samples: list of (inputs_embeds [T_i, H], labels [T_i]) -> right-padded\n    (inputs_embeds [B, T_max, H], labels [B, T_max] filled with -100,\n    attention_mask [B, T_max])."""\n    max_len = max(e.shape[0] for e, _ in samples)\n    hidden = samples[0][0].shape[1]\n    device = samples[0][0].device\n    dtype = samples[0][0].dtype\n    b = len(samples)\n    embeds = torch.zeros(b, max_len, hidden, dtype=dtype, device=device)\n    labels = torch.full((b, max_len), -100, dtype=torch.long, device=device)\n    attn = torch.zeros(b, max_len, dtype=torch.long, device=device)\n    for i, (e, lab) in enumerate(samples):\n        n = e.shape[0]\n        embeds[i, :n] = e\n        labels[i, :n] = lab\n        attn[i, :n] = 1\n    return embeds, labels, attn\n\n\nclass BitNetVLM(nn.Module):\n    def __init__(self, llm, tokenizer, vision_adapter, max_text_len: int = 256, placeholder: str = IMAGE_PLACEHOLDER):\n        """`vision_adapter` is any module shaped like VisionAdapter/AudioAdapter:\n        callable as `adapter(features)` (or `adapter(*features)` when\n        `features` is a tuple -- see `encode`), returning either a\n        [B, N, llm_hidden] tensor or a length-B list of [N_i, llm_hidden]\n        tensors, plus a `.trainable_parameters()` method. The parameter/\n        attribute name stays `vision_adapter` (not renamed to `adapter`) so\n        every existing eyes caller (train_stage1.py, train_stage2.py,\n        BitNetVLMStage2) that reads `vlm.vision_adapter.*` keeps working\n        unchanged; `self.adapter` below is a plain alias to the SAME object\n        for generic/modality-agnostic code (audio_adapter\'s ears path).\n        `placeholder` is the marker `_split_prompt` looks for in the prompt\n        text -- "<image>" by default (eyes), pass AUDIO_PLACEHOLDER\n        ("<audio>") for the ears scripts."""\n        super().__init__()\n        self.llm = llm\n        self.tokenizer = tokenizer\n        self.vision_adapter = vision_adapter\n        self.adapter = vision_adapter\n        self.max_text_len = max_text_len\n        self.placeholder = placeholder\n\n        for p in self.llm.parameters():\n            p.requires_grad = False\n        self.llm.eval()\n        self.embed_tokens = self.llm.get_input_embeddings()\n\n        eot = tokenizer.convert_tokens_to_ids("<|eot_id|>")\n        self.eot_id = eot if isinstance(eot, int) and eot >= 0 else tokenizer.eos_token_id\n\n    def train(self, mode: bool = True):\n        super().train(mode)\n        self.llm.eval()  # always frozen/eval: no dropout drift in a model we never update\n        return self\n\n    def _encode_text(self, text: str) -> list:\n        if not text:\n            return []\n        ids = self.tokenizer(text, add_special_tokens=False)["input_ids"]\n        return ids[: self.max_text_len]\n\n    def _embed_ids(self, ids: list, like: torch.Tensor) -> torch.Tensor:\n        if not ids:\n            return like.new_zeros((0, like.shape[-1]))\n        idx = torch.tensor(ids, dtype=torch.long, device=like.device)\n        return self.embed_tokens(idx)\n\n    def _split_prompt(self, prompt: str) -> tuple:\n        if self.placeholder not in prompt:\n            prompt = self.placeholder + "\\n" + prompt\n        before, after = prompt.split(self.placeholder, 1)\n        return before, after\n\n    def encode(self, features):\n        """Generic modality encode: `features` -> [B, N, llm_hidden] tensor,\n        or a length-B list of [N_i, llm_hidden] tensors (a variable-length\n        modality, e.g. audio -- see audio_adapter.AudioAdapter.forward).\n        Calls whichever adapter this instance was built with. `features` is\n        normally a single tensor (VisionAdapter\'s pixel_values); pass a\n        tuple to splat multiple positional args into the adapter (e.g.\n        AudioAdapter.forward\'s (input_features, num_frames))."""\n        if isinstance(features, tuple):\n            return self.adapter(*features)\n        return self.adapter(features)\n\n    def encode_images(self, pixel_values: torch.Tensor) -> torch.Tensor:\n        """pixel_values: [B,3,H,W] -> [B, tokens_per_image, llm_hidden].\n        Name-stable alias of `encode` kept for the eyes scripts/tests\n        (train_stage1.py, train_stage2.py\'s BitNetVLMStage2) -- do not\n        remove or change its signature."""\n        return self.encode(pixel_values)\n\n    def _prefix_embeds(self, prompt: str, img_embeds_i: torch.Tensor) -> torch.Tensor:\n        before, after = self._split_prompt(prompt)\n        header_ids = self._encode_text(f"User: {before}")\n        tail_ids = self._encode_text(after.rstrip() + "<|eot_id|>Assistant: ")\n        header_emb = self._embed_ids(header_ids, img_embeds_i)\n        tail_emb = self._embed_ids(tail_ids, img_embeds_i)\n        return torch.cat([header_emb, img_embeds_i, tail_emb], dim=0)\n\n    def build_sample(self, prompt: str, answer: str, img_embeds_i: torch.Tensor):\n        """One (prompt, answer, image) triple -> (inputs_embeds [T,H], labels [T])\n        with labels = -100 everywhere except the answer tokens."""\n        prefix = self._prefix_embeds(prompt, img_embeds_i)\n        answer_ids = self._encode_text(answer.strip() + "<|eot_id|>")\n        answer_emb = self._embed_ids(answer_ids, img_embeds_i)\n        inputs_embeds = torch.cat([prefix, answer_emb], dim=0)\n        labels = torch.tensor([-100] * prefix.shape[0] + answer_ids, dtype=torch.long, device=prefix.device)\n        return inputs_embeds, labels\n\n    def forward(self, prompts: list, answers: list, features):\n        """prompts/answers: parallel lists of str, one per sample. features:\n        whatever `encode` accepts for this instance\'s adapter -- [B,3,H,W]\n        pixel_values (one image per sample, VisionAdapter) or an\n        (input_features, num_frames) tuple (AudioAdapter). Returns\n        (loss, logits) from the underlying HF CausalLM forward. Parameter\n        was named `pixel_values` before this became modality-generic;\n        renamed here since every call site passes it positionally."""\n        embeds_per_sample = self.encode(features)  # [B, N, hidden] tensor, or a length-B list of [N_i, hidden]\n        samples = [self.build_sample(p, a, embeds_per_sample[i]) for i, (p, a) in enumerate(zip(prompts, answers))]\n        embeds, labels, attn = pad_and_stack(samples)\n        out = self.llm(inputs_embeds=embeds, attention_mask=attn, labels=labels)\n        return out.loss, out.logits\n\n    def _greedy_generate(self, prefix: torch.Tensor, max_new_tokens: int) -> str:\n        """prefix: [1,T,H] inputs_embeds for one sample, already unsqueezed\n        to batch size 1. Shared tail of generate_caption/\n        generate_from_features: greedy decode via the HF `generate()`\n        machinery (KV-cached), seeded with the spliced prefix\'s\n        inputs_embeds -- only the newly generated tokens come back when\n        generate() is started from inputs_embeds rather than input_ids."""\n        # generate() runs outside autocast: inputs_embeds must match the LLM\'s own dtype (bf16 on Colab).\n        prefix = prefix.to(self.llm.get_input_embeddings().weight.dtype)\n        attn = torch.ones(prefix.shape[:2], dtype=torch.long, device=prefix.device)\n        pad_id = self.tokenizer.pad_token_id\n        if pad_id is None:\n            pad_id = self.eot_id\n        out_ids = self.llm.generate(inputs_embeds=prefix, attention_mask=attn, max_new_tokens=max_new_tokens,\n                                     do_sample=False, eos_token_id=self.eot_id, pad_token_id=pad_id)\n        return self.tokenizer.decode(out_ids[0], skip_special_tokens=True)\n\n    @torch.no_grad()\n    def generate_caption(self, pixel_values_1: torch.Tensor, prompt: str = "<image>\\nDescribe the image briefly.",\n                          max_new_tokens: int = 20) -> str:\n        """pixel_values_1: [3,H,W], a single image (unbatched -- this method\n        adds the batch dim itself). Unchanged behaviour from before this\n        module became modality-generic; see `generate_from_features` for\n        the audio/generic equivalent (audio preprocessing already returns a\n        batch dim even for one clip, so it is not unsqueezed here)."""\n        was_training = self.training\n        self.eval()\n        img = self.encode_images(pixel_values_1.unsqueeze(0))[0]\n        prefix = self._prefix_embeds(prompt, img).unsqueeze(0)\n        text = self._greedy_generate(prefix, max_new_tokens)\n        if was_training:\n            self.train()\n        return text\n\n    @torch.no_grad()\n    def generate_from_features(self, features_1, prompt: str, max_new_tokens: int = 20) -> str:\n        """Like generate_caption but for a single ALREADY-BATCHED-TO-ONE\n        sample of any modality this instance\'s adapter accepts -- e.g. an\n        (input_features [1,n_mels,T], num_frames [1]) tuple for AudioAdapter\n        (see audio_adapter.py\'s `prepare_features`, which always returns a\n        batch dim, even for a length-1 list of clips). Unlike\n        generate_caption, `features_1` is NOT unsqueezed here -- the\n        caller\'s own batch-of-one preprocessing already has it. Used by\n        train_stage1_audio.py\'s eval; train_stage1.py/train_stage2.py keep\n        using generate_caption unchanged."""\n        was_training = self.training\n        self.eval()\n        img = self.encode(features_1)[0]\n        prefix = self._prefix_embeds(prompt, img).unsqueeze(0)\n        text = self._greedy_generate(prefix, max_new_tokens)\n        if was_training:\n            self.train()\n        return text\n\n\n# ---------------------------------------------------------------------------\n# LLM builders\n# ---------------------------------------------------------------------------\n\ndef load_real_bitnet_llm(hf_dir: str):\n    """Loads the real 2.4B offline-quantized microsoft/bitnet-b1.58-2B-4T\n    checkpoint, applying the same unmaterialized-weight fix\n    forge/bitnet_reference.py uses -- see that module\'s\n    _fix_unmaterialized_offline_weights docstring for why it is needed on\n    this transformers version. The LLM stays frozen for the whole of stage\n    1 (see BitNetVLM.__init__), so loading it in float32 first (required by\n    the fix\'s numpy round-trip) and letting the caller cast the frozen\n    copy to bf16 afterward is fine -- there is no optimizer state riding on\n    these weights."""\n    sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(__file__)), ".."))\n    from bitnet_reference import _fix_unmaterialized_offline_weights  # noqa: E402\n    from transformers import AutoModelForCausalLM  # noqa: E402\n\n    model = AutoModelForCausalLM.from_pretrained(hf_dir, torch_dtype=torch.float32, low_cpu_mem_usage=True)\n    n_fixed = _fix_unmaterialized_offline_weights(model, hf_dir)\n    print(f"[mm_model] unpacked {n_fixed} offline AutoBitLinear modules from {hf_dir}")\n    model.eval()\n    return model\n\n\ndef build_tiny_bitnet_llm(hidden: int, vocab: int, bos_id: int, eos_id: int,\n                           layers: int = 2, heads: int = 4, kv_heads: int = 2,\n                           ffn: int = 96, max_pos: int = 256):\n    """A randomly-initialized, online-quantized BitNet stand-in LLM for\n    train_stage1.py --smoke (no checkpoint, no download) -- same\n    construction forge/bitnet_reference.py\'s run_tiny() uses for its\n    equivalence-test fixture. `vocab`/`bos_id`/`eos_id` are normally the\n    REAL BitNet tokenizer\'s (already local, no extra download), so the\n    smoke path exercises the actual chat-template tokenization end to end;\n    only the transformer body (hidden/layers/heads/ffn) is shrunk."""\n    from transformers.integrations.bitnet import AutoBitLinear, replace_with_bitnet_linear\n    from transformers.models.bitnet.configuration_bitnet import BitNetConfig as HFBitNetConfig\n    from transformers.models.bitnet.modeling_bitnet import BitNetForCausalLM\n\n    cfg = HFBitNetConfig(\n        vocab_size=vocab, hidden_size=hidden, intermediate_size=ffn,\n        num_hidden_layers=layers, num_attention_heads=heads, num_key_value_heads=kv_heads,\n        max_position_embeddings=max_pos, rope_theta=10000.0, rms_norm_eps=1e-5,\n        tie_word_embeddings=True, attention_bias=False,\n        bos_token_id=bos_id, eos_token_id=eos_id,\n    )\n    model = BitNetForCausalLM(cfg)\n\n    class _QC:\n        linear_class = "autobitlinear"\n        quantization_mode = "online"\n        use_rms_norm = False\n        rms_norm_eps = 1e-5\n\n    replace_with_bitnet_linear(model, modules_to_not_convert=["lm_head"], quantization_config=_QC())\n    # As in bitnet_reference.run_tiny: replace_with_bitnet_linear builds the\n    # new AutoBitLinear modules on the meta device (designed for a\n    # from_pretrained load that materializes them afterward); since there is\n    # no checkpoint here, materialize real random weights before any forward.\n    for mod in model.modules():\n        if isinstance(mod, AutoBitLinear):\n            w = torch.empty(mod.weight.shape, dtype=torch.float32)\n            w.normal_(mean=0.0, std=1.0)\n            mod.weight = torch.nn.Parameter(w, requires_grad=False)\n    model.eval()\n    return model, cfg\n', 'forge/multimodal/lora_bitlinear.py': '"""forge/multimodal/lora_bitlinear.py -- LoRA adapters for the BitNet LLM\'s\n`transformers.integrations.bitnet.AutoBitLinear` projections, for stage 2\ninstruction tuning (train_stage2.py). Read this alongside\n`transformers/integrations/bitnet.py` (AutoBitLinear, WeightQuant, ActQuant)\nand `transformers/models/bitnet/modeling_bitnet.py` (BitNetAttention/\nBitNetMLP, which is where q_proj/k_proj/v_proj/o_proj/gate_proj/up_proj/\ndown_proj actually live) -- this module assumes, and does not re-derive,\ntheir forward contracts.\n\nTwo frozen-base variants (`--base offline|bf16` in train_stage2.py; the\nloader here is `load_frozen_base`):\n\n  "offline" (DEFAULT, preferred for this project\'s pipeline) --\n      microsoft/bitnet-b1.58-2B-4T, the same checkpoint mm_model.\n      load_real_bitnet_llm loads for stage 1. Its AutoBitLinear modules have\n      `online_quant=False`: `.weight` holds ALREADY-TERNARY values (-1/0/1,\n      unpacked from the checkpoint\'s 2-bit-packed uint8 tensors -- see\n      mm_model.load_real_bitnet_llm\'s docstring for the from_pretrained\n      unmaterialization workaround this project needs on this transformers\n      version) plus a scalar `.weight_scale`; forward = ActQuant(x) @\n      weight^T, rescaled by weight_scale. 1.2 GB on disk, and -- critically\n      -- the EXACT SAME weights cortex/bitnet.go\'s Go engine will run, so a\n      LoRA delta trained against this base composes correctly with the Go\n      engine\'s own ternary forward at inference time with no train/serve\n      skew. No STE noise either (the ternary weights are fixed data, not a\n      quantize-every-forward simulation). Preferred for this project.\n  "bf16" (fallback, Colab-only; NOT exercised on this PC -- downloading it\n      would exceed this PC\'s 50 MB-per-file budget, see AGENTS.md) --\n      microsoft/bitnet-b1.58-2B-4T-bf16, the trainable checkpoint. Its\n      config.json declares `quantization_mode: online`, so `from_pretrained`\n      still converts every target nn.Linear to AutoBitLinear (transformers/\n      quantizers/quantizer_bitnet.py dispatches on the checkpoint\'s own\n      quantization_config, unconditionally -- see that file), but with\n      `online_quant=True`: `.weight` is a REAL bf16 nn.Parameter, and every\n      forward pass re-ternarizes it on the fly via WeightQuant\'s\n      straight-through estimator (transformers/integrations/bitnet.py).\n      This lets the BASE itself move (useful for a heavier full-finetune\n      later), but the ternary weights it re-quantizes to on any given step\n      are a moving target that will, in general, NOT match whatever the\n      offline checkpoint\'s fixed packed weights are -- so a LoRA delta\n      trained against this base does not obviously transfer to the Go\n      engine\'s frozen offline weights without re-exporting the base itself\n      too (out of scope here: this project only ever ships the LoRA delta,\n      never a merged base -- see `merge_lora`). Costs ~4.8 GB instead of\n      1.2 GB and adds STE gradient noise to every base matmul. Only use this\n      if a future experiment specifically wants the base to keep moving.\n\nLoRA math, and WHERE the delta enters (the task\'s explicit design question):\n`LoRABitLinear.forward(x) = base(x) + (alpha/r) * B(A(dropout(x)))`, where\n`base(x)` is the wrapped AutoBitLinear\'s own, unmodified `forward(x)` (so it\napplies its own internal `ActQuant.apply(x)` to whatever copy of `x` it\nreceives), and the LoRA branch reads the SAME raw `x` this module\'s forward\nreceives -- i.e. the activation BEFORE AutoBitLinear\'s internal ActQuant, not\na quantized copy of it. Concretely, LoRABitLinear never calls ActQuant\nitself; it just hands `x` to `self.base(x)` (which quantizes its own copy\ninternally) and, separately, computes the delta straight off the same `x`.\nThis is standard QLoRA-style practice, for two reasons: (1) re-quantizing x\na second time for the LoRA path would throw away exactly the extra\nprecision the low-rank adapter exists to add back, since the base path\nalready lost most of it to ActQuant\'s 8-bit per-token quantization; (2) it\navoids invoking ActQuant\'s own `@torch.compile`-wrapped forward twice per\nlayer for no benefit. The delta matmuls themselves are NOT quantized at\nall -- they run in whatever dtype `x` and the LoRA A/B parameters are in.\nA/B are kept as fp32 master weights (`lora_A`, `lora_B`), the same\noptimizer-stability convention train_stage1.py documents for the projector\n("trainable projector stays fp32 ... autocast during the forward pass");\nunder Colab\'s `torch.autocast(..., dtype=bf16)` (train_stage2.py) the delta\nmatmuls run in bf16 like everything else in that context, and on the PC\'s\n--smoke path (no autocast) they run in plain fp32.\n\nTarget modules (`DEFAULT_TARGETS`, matching the task spec and\nBitNetAttention/BitNetMLP\'s own attribute names): q_proj, k_proj, v_proj,\no_proj, gate_proj, up_proj, down_proj -- all 7 per layer, 30 layers on the\nreal 2.4B checkpoint (210 AutoBitLinear modules total, same count\nmm_model.load_real_bitnet_llm\'s docstring cites for the unmaterialization\nfix).\n"""\n\nfrom __future__ import annotations\n\nimport math\nimport os\nimport re\nimport sys\nfrom typing import Iterable\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\nDEFAULT_TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]\n\n# "model.layers.<i>.self_attn.q_proj" / "model.layers.<i>.mlp.gate_proj" --\n# confirmed against transformers/models/bitnet/modeling_bitnet.py\'s\n# BitNetModel (self.layers = nn.ModuleList(...)) and BitNetForCausalLM\n# (self.model = BitNetModel(config)): named_modules() on the top-level LLM\n# yields exactly this shape for every target projection, real checkpoint or\n# the --smoke tiny stand-in alike (both are BitNetForCausalLM instances).\nLAYER_PROJ_RE = re.compile(r"\\.layers\\.(\\d+)\\.(?:self_attn|mlp)\\.(\\w+)$")\n\n\ndef _autobitlinear_cls():\n    from transformers.integrations.bitnet import AutoBitLinear\n    return AutoBitLinear\n\n\nclass LoRABitLinear(nn.Module):\n    """Wraps one frozen `AutoBitLinear` `base` with a trainable low-rank\n    delta. See the module docstring for the exact forward math and where the\n    delta enters relative to `base`\'s own ActQuant. `base` itself is frozen\n    here (`requires_grad_(False)` on every one of its parameters) -- callers\n    should not rely on it having been frozen already."""\n\n    def __init__(self, base: nn.Module, r: int = 16, alpha: int = 32, dropout: float = 0.0):\n        super().__init__()\n        AutoBitLinear = _autobitlinear_cls()\n        if not isinstance(base, AutoBitLinear):\n            raise TypeError(f"LoRABitLinear expects an AutoBitLinear base, got {type(base).__name__}")\n        if r <= 0:\n            raise ValueError(f"r must be positive, got {r}")\n        self.base = base\n        for p in self.base.parameters():\n            p.requires_grad_(False)\n\n        self.in_features = base.in_features\n        self.out_features = base.out_features\n        self.r = r\n        self.alpha = alpha\n        self.scaling = alpha / r\n        self.lora_dropout = nn.Dropout(dropout) if dropout > 0 else nn.Identity()\n\n        # fp32 master weights -- see module docstring\'s "LoRA math" section\n        # for why (matches train_stage1.py\'s projector convention).\n        self.lora_A = nn.Parameter(torch.empty(r, self.in_features, dtype=torch.float32))\n        self.lora_B = nn.Parameter(torch.zeros(self.out_features, r, dtype=torch.float32))\n        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))\n        # lora_B intentionally left at zero-init (standard LoRA, Hu et al.\n        # 2021): the delta contributes exactly nothing until training moves\n        # B away from zero, so a freshly injected model\'s forward is\n        # bit-identical to the un-adapted base (see\n        # tests/test_lora_bitlinear.py\'s test_zero_b_matches_base).\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        base_out = self.base(x)\n        # `x` here is the pre-ActQuant activation -- see module docstring.\n        lora_in = self.lora_dropout(x).to(self.lora_A.dtype)\n        delta = F.linear(F.linear(lora_in, self.lora_A), self.lora_B) * self.scaling\n        return base_out + delta.to(base_out.dtype)\n\n    def extra_repr(self) -> str:\n        return f"in_features={self.in_features}, out_features={self.out_features}, r={self.r}, alpha={self.alpha}"\n\n\ndef _iter_target_leaves(model: nn.Module, targets: Iterable[str]):\n    AutoBitLinear = _autobitlinear_cls()\n    target_set = set(targets)\n    for name, module in model.named_modules():\n        leaf = name.rsplit(".", 1)[-1]\n        if leaf in target_set and isinstance(module, AutoBitLinear):\n            yield name, module\n\n\ndef inject_lora(model: nn.Module, targets: Iterable[str] | None = None, r: int = 16, alpha: int = 32,\n                 dropout: float = 0.0) -> list:\n    """Replaces every `AutoBitLinear` submodule of `model` whose leaf\n    attribute name is in `targets` with a `LoRABitLinear` wrapping it, IN\n    PLACE (the original module -- and its already-loaded weights/\n    weight_scale/online_quant flag -- becomes the new module\'s `.base`, so\n    the exact frozen weights the caller loaded keep being used; only the new\n    A/B matrices are added on top). Returns the flat list of newly created\n    trainable `nn.Parameter`s (`lora_A`, `lora_B` for every wrapped module,\n    in injection order) -- pass straight to an optimizer param group.\n\n    Raises `ValueError` if no matching `AutoBitLinear` modules are found\n    (almost always a sign `targets` doesn\'t match this model\'s attribute\n    names, or `model` was never converted to AutoBitLinear in the first\n    place -- see `load_frozen_base`)."""\n    targets = list(targets or DEFAULT_TARGETS)\n    hits = list(_iter_target_leaves(model, targets))\n    if not hits:\n        raise ValueError(f"inject_lora found no AutoBitLinear modules matching targets={targets} in this model")\n    trainable: list = []\n    for name, module in hits:\n        wrapped = LoRABitLinear(module, r=r, alpha=alpha, dropout=dropout)\n        parent_name, _, leaf = name.rpartition(".")\n        parent = model.get_submodule(parent_name) if parent_name else model\n        setattr(parent, leaf, wrapped)\n        trainable.append(wrapped.lora_A)\n        trainable.append(wrapped.lora_B)\n    return trainable\n\n\ndef lora_modules(model: nn.Module) -> dict:\n    """{dotted module path: LoRABitLinear} for every LoRABitLinear currently\n    in `model` (i.e. whatever `inject_lora` created)."""\n    return {name: mod for name, mod in model.named_modules() if isinstance(mod, LoRABitLinear)}\n\n\ndef merge_lora(model: nn.Module) -> None:\n    """NOT SUPPORTED, by design -- always raises. Merging `B @ A` into\n    `base.weight` is not a meaningful operation for either `--base` variant\n    this project uses:\n\n      - offline: `base.weight` holds ALREADY-TERNARY values (-1/0/1) plus a\n        single scalar `weight_scale`. `B @ A` is a dense, full-precision\n        matrix; adding it to the ternary weight and re-quantizing back to\n        {-1,0,1} would throw away almost everything the LoRA update was\n        trained to add (unlike merging at full precision, this is very much\n        NOT a no-op) -- and even the un-requantized dense sum could no\n        longer be packed into the 2-bit ternary representation the Go\n        engine (cortex/bitnet.go / bitnet_linear.go) and the exported NXTF\n        format require.\n      - bf16: `base.weight` is a real bf16 Parameter, so merging is at least\n        numerically possible there -- but is intentionally not implemented\n        either, to keep both `--base` variants\' export contract identical:\n        stage 2 always ships the LoRA delta as SEPARATE A/B tensors\n        (export_stage2.py), and the Go engine applies\n        `y = base(x) + (alpha/r) * B(A(x))` itself at inference time. A\n        merge helper here would invite a second, divergent export path.\n\n    Use `lora_state_dict()` / `export_stage2.py` to ship the LoRA delta\n    alongside the frozen base instead."""\n    raise NotImplementedError(\n        "merge_lora() is not supported: the frozen base\'s AutoBitLinear.weight is either already-ternary "\n        "(offline -- merging would destroy the ternary representation) or intentionally kept separate from "\n        "the LoRA delta (bf16 -- see export_stage2.py, which the Go engine\'s inference-time delta application "\n        "depends on). Use lora_state_dict()/export_stage2.py to export A/B instead."\n    )\n\n\ndef lora_state_dict(model: nn.Module) -> dict:\n    """{dotted module path: {"A": Tensor[r,in], "B": Tensor[out,r], "r": int,\n    "alpha": int}} for every LoRABitLinear in `model`, detached and moved to\n    CPU (safe to `torch.save` directly, e.g. inside train_stage2.py\'s\n    checkpoint dict)."""\n    out = {}\n    for name, mod in lora_modules(model).items():\n        out[name] = {\n            "A": mod.lora_A.detach().cpu().clone(),\n            "B": mod.lora_B.detach().cpu().clone(),\n            "r": mod.r,\n            "alpha": mod.alpha,\n        }\n    return out\n\n\ndef load_lora_state_dict(model: nn.Module, state: dict, strict: bool = True) -> int:\n    """Loads a dict shaped like `lora_state_dict()`\'s output into `model`\'s\n    EXISTING LoRABitLinear modules in place (`.data.copy_`, so each\n    Parameter keeps its own device/dtype -- `state`\'s tensors are cast to\n    match). Returns the number of modules loaded.\n\n    `strict=True` (default) raises `KeyError` if `state` names a module path\n    `model` doesn\'t have a LoRABitLinear at (or vice versa) -- almost always\n    a sign the caller injected LoRA with different `targets`/layer count\n    than whatever produced `state`."""\n    modules = lora_modules(model)\n    if strict:\n        missing = set(modules) - set(state)\n        unexpected = set(state) - set(modules)\n        if missing or unexpected:\n            raise KeyError(f"lora_state_dict mismatch: missing={sorted(missing)} unexpected={sorted(unexpected)}")\n    n = 0\n    for name, entry in state.items():\n        mod = modules.get(name)\n        if mod is None:\n            continue\n        if mod.lora_A.shape != entry["A"].shape or mod.lora_B.shape != entry["B"].shape:\n            raise ValueError(f"{name}: shape mismatch loading LoRA state (A {mod.lora_A.shape} vs {entry[\'A\'].shape})")\n        mod.lora_A.data.copy_(entry["A"].to(device=mod.lora_A.device, dtype=mod.lora_A.dtype))\n        mod.lora_B.data.copy_(entry["B"].to(device=mod.lora_B.device, dtype=mod.lora_B.dtype))\n        n += 1\n    return n\n\n\ndef export_key(module_path: str) -> str:\n    """"model.layers.3.self_attn.q_proj" -> "lora.layers.3.q_proj" -- the\n    tensor-name convention export_stage2.py writes (`.A`/`.B` appended by\n    the caller). Shared here so train_stage2.py and export_stage2.py (and\n    any future Go importer) agree on exactly one naming scheme instead of\n    each re-deriving it."""\n    m = LAYER_PROJ_RE.search(module_path)\n    if not m:\n        raise ValueError(f"export_key: can\'t parse a \'...layers.<i>.(self_attn|mlp).<proj>\' suffix out of {module_path!r}")\n    layer_idx, proj = m.group(1), m.group(2)\n    return f"lora.layers.{layer_idx}.{proj}"\n\n\ndef load_frozen_base(base: str, llm_dir: str):\n    """`base`: "offline" | "bf16" -> a frozen (`requires_grad=False` on\n    every parameter), `eval()`-mode BitNetForCausalLM ready for\n    `inject_lora`. See the module docstring\'s "Two frozen-base variants"\n    section for the trade-off this flag controls. `llm_dir` is a local\n    directory (this project never downloads a checkpoint mid-run -- see\n    AGENTS.md); for "bf16" that means a LOCAL copy of\n    microsoft/bitnet-b1.58-2B-4T-bf16, which this PC does not have (~4.8 GB,\n    over the 50 MB/file budget) -- that path is Colab-only and untested\n    here; "offline" is data/pretrained/bitnet-b1.58-2B-4T, already local."""\n    if base == "offline":\n        # Reuse (never modify) mm_model\'s loader -- it already carries the\n        # from_pretrained unmaterialization workaround this checkpoint\n        # needs on this transformers version (see its own docstring).\n        sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))\n        from mm_model import load_real_bitnet_llm  # noqa: E402\n\n        model = load_real_bitnet_llm(llm_dir)\n    elif base == "bf16":\n        from transformers import AutoModelForCausalLM\n\n        model = AutoModelForCausalLM.from_pretrained(llm_dir, torch_dtype=torch.bfloat16, low_cpu_mem_usage=True)\n        model.eval()\n    else:\n        raise ValueError(f"load_frozen_base: base must be \'offline\' or \'bf16\', got {base!r}")\n\n    for p in model.parameters():\n        p.requires_grad_(False)\n    return model\n', 'forge/test_tool_data.py': 'import json\nfrom pathlib import Path\nimport tempfile\nimport unittest\nfrom forge.tool_data import load_trajectories, validate_splits, encode_trajectory\n\n\ndef row(prompt, lang="en"):\n    return {"language": lang, "messages": [\n        {"role": "system", "content": "Answer in English."},\n        {"role": "user", "content": prompt},\n        {"role": "assistant", "content": "CALL calc: 2+3"},\n        {"role": "tool", "content": "5"},\n        {"role": "assistant", "content": "The answer is 5."}]}\n\n\nclass ToolDataTests(unittest.TestCase):\n    def test_generation_header_boundary_is_preserved(self):\n        class MergingTokenizer:\n            def encode(self, text, add_special_tokens=False):\n                # Simulate real BPE: the same visible space has different IDs\n                # when encoded with the answer instead of the generation header.\n                if text == "Assistant: ":\n                    return [10, 11, 12]\n                if text.startswith("Assistant: "):\n                    return [10, 11, 99]\n                if text.startswith("CALL "):\n                    return [20, 21, 22]\n                if text.startswith("The answer"):\n                    return [30, 31, 22]\n                return [40, 22]\n        encoded = encode_trajectory(row("Compute"), MergingTokenizer(), 100)\n        self.assertEqual(encoded["input_ids"], [40,22,40,22,10,11,12,20,21,22,40,22,10,11,12,30,31,22])\n        self.assertEqual([x for x in encoded["labels"] if x != -100], [20,21,22,30,31,22])\n\n    def test_language_and_tool_observation_validation(self):\n        with tempfile.TemporaryDirectory() as d:\n            p = Path(d) / "rows.jsonl"\n            p.write_text(json.dumps(row("Compute two plus three")), encoding="utf-8")\n            self.assertEqual(len(load_trajectories(p)), 1)\n            p.write_text(json.dumps(row("Calculează", "ro")), encoding="utf-8")\n            with self.assertRaises(ValueError):\n                load_trajectories(p)\n            self.assertEqual(len(load_trajectories(p, ("en", "ro"))), 1)\n            bad = row("hi")\n            del bad["messages"][3]\n            p.write_text(json.dumps(bad), encoding="utf-8")\n            with self.assertRaises(ValueError):\n                load_trajectories(p)\n\n    def test_overlap_and_assistant_mask(self):\n        with self.assertRaises(ValueError):\n            validate_splits([row("Hello")], [row("HELLO", "ro")])\n        en, ro = row("Hello"), row("Salut", "ro")\n        en["task_id"] = ro["task_id"] = "greeting-1"\n        with self.assertRaises(ValueError):\n            validate_splits([en], [ro])\n        class Tok:\n            def encode(self, text, add_special_tokens=False):\n                return list(text.encode("utf-8"))\n        encoded = encode_trajectory(row("hi"), Tok(), 1024)\n        supervised = bytes(i for i in encoded["labels"] if i != -100).decode()\n        self.assertIn("CALL calc: 2+3", supervised)\n        self.assertNotIn("Answer in English", supervised)\n        self.assertNotIn("Tool:", supervised)\n        with self.assertRaises(ValueError):\n            encode_trajectory(row("hi"), Tok(), 5)\n', 'forge/test_train_tools.py': '"""End-to-end synthetic SFT, export and exact interrupted/resumed training."""\nimport json\nimport itertools\nfrom pathlib import Path\nimport sys\nimport tempfile\nimport unittest\nfrom unittest.mock import patch\nfrom forge.test_tool_data import row\n\n\nclass TrainerTests(unittest.TestCase):\n    def test_warm_start_export_and_reject_corrupt_tensors(self):\n        from forge import train_tools\n        import torch\n        import lora_bitlinear as lb\n        from mm_model import build_tiny_bitnet_llm\n        from safetensors.torch import save_file\n        base, _ = build_tiny_bitnet_llm(16, 64, 0, 1, layers=1, heads=2, kv_heads=1, ffn=32, max_pos=512)\n        lb.inject_lora(base, r=2, alpha=4)\n        modules = lb.lora_modules(base)\n        tensors = {lb.export_key(name) + "." + key: torch.ones_like(parameter).cpu().contiguous()\n                   for name, module in modules.items()\n                   for key, parameter in (("A", module.lora_A), ("B", module.lora_B))}\n        with tempfile.TemporaryDirectory() as d:\n            prefix = str(Path(d) / "init")\n            Path(prefix + ".json").write_text(json.dumps({"base": {"kind": "smoke"},\n                "lora": {"r": 2, "alpha": 4, "scaling": 2, "n_modules": len(modules)}}))\n            save_file(tensors, prefix + ".safetensors")\n            provenance = train_tools.initialize_adapter(base, prefix, lb, "smoke", 2, 4)\n            self.assertEqual(len(provenance["weights_sha256"]), 64)\n            self.assertTrue(all(torch.equal(m.lora_B, torch.ones_like(m.lora_B)) for m in modules.values()))\n            with self.assertRaisesRegex(ValueError, "base kind"):\n                train_tools.initialize_adapter(base, prefix, lb, "offline", 2, 4)\n            first = next(iter(tensors))\n            tensors[first].fill_(float("nan"))\n            save_file(tensors, prefix + ".safetensors")\n            with self.assertRaisesRegex(ValueError, "invalid initial adapter"):\n                train_tools.initialize_adapter(base, prefix, lb, "smoke", 2, 4)\n\n    def test_resume_matches_uninterrupted_and_exports_adapters(self):\n        from forge import train_tools\n        import torch\n        torch.set_num_threads(1)\n        with tempfile.TemporaryDirectory() as d:\n            root = Path(d)\n            train, val = root / "train.jsonl", root / "val.jsonl"\n            train.write_text("\\n".join(json.dumps(row(f"Compute test {i}")) for i in range(4)), encoding="utf-8")\n            val.write_text(json.dumps(row("Validation question")), encoding="utf-8")\n            def run(out, resume=False):\n                args = ["train_tools", "--smoke", "--train", str(train), "--validation", str(val),\n                        "--llm-dir", "unused-smoke-model", "--out", str(out), "--steps", "2", "--batch", "1",\n                        "--accum", "2", "--rank", "2", "--alpha", "4", "--max-length", "512", "--checkpoint-every", "1", "--gradient-checkpointing"]\n                if resume:\n                    args.append("--resume")\n                with patch.object(sys, "argv", args):\n                    train_tools.main()\n            uninterrupted, resumed = root / "full", root / "resumed"\n            run(uninterrupted)\n            with patch("safetensors.torch.save_file", side_effect=RuntimeError("injected interruption")):\n                with self.assertRaisesRegex(RuntimeError, "injected interruption"):\n                    run(resumed)\n            ck = torch.load(resumed / "checkpoint.pt", weights_only=False)\n            self.assertEqual(ck["step"], 1)\n            run(resumed, resume=True)\n            expected = torch.load(uninterrupted / "checkpoint.pt", weights_only=False)\n            actual = torch.load(resumed / "checkpoint.pt", weights_only=False)\n            self.assertEqual(actual["step"], 2)\n            changed = False\n            for name, entry in actual["lora"].items():\n                for key in ("A", "B"):\n                    torch.testing.assert_close(entry[key], expected["lora"][name][key], rtol=0, atol=0)\n                changed |= bool(entry["B"].abs().sum())\n            self.assertTrue(changed, "training did not update adapters")\n            self.assertTrue((resumed / "adapter-step2.safetensors").is_file())\n            meta = json.loads((resumed / "adapter-step2.json").read_text())\n            self.assertEqual(meta["base"]["kind"], "smoke")\n            self.assertTrue((uninterrupted / "baseline.json").is_file())\n\n    def test_soft_deadline_saves_before_last_step(self):\n        from forge import train_tools\n        import torch\n        torch.set_num_threads(1)\n        with tempfile.TemporaryDirectory() as d:\n            root = Path(d)\n            train, val = root / "train.jsonl", root / "val.jsonl"\n            train.write_text(json.dumps(row("Train question")), encoding="utf-8")\n            val.write_text(json.dumps(row("Validation question")), encoding="utf-8")\n            out = root / "run"\n            args = ["train_tools", "--smoke", "--train", str(train), "--validation", str(val),\n                    "--llm-dir", "smoke", "--out", str(out), "--steps", "3", "--accum", "1",\n                    "--max-length", "512", "--checkpoint-every", "3", "--max-runtime-minutes", "0.000000001"]\n            with patch.object(sys, "argv", args), patch("forge.train_tools.time.monotonic", side_effect=itertools.count().__next__):\n                train_tools.main()\n            ck = torch.load(out / "checkpoint.pt", weights_only=False)\n            self.assertEqual(ck["step"], 1)\n            self.assertTrue((out / "adapter-step1.json").is_file())\n', 'forge/test_distributed_tools.py': '"""Two-worker CPU DDP smoke test; no models, corpora or GPU allocation."""\nimport json\nimport os\nfrom pathlib import Path\nimport socket\nimport subprocess\nimport sys\nimport tempfile\nimport unittest\nfrom forge.test_tool_data import row\n\n\nclass DistributedTrainerTests(unittest.TestCase):\n    def test_two_workers_export_one_checkpoint(self):\n        with tempfile.TemporaryDirectory() as d:\n            root = Path(d)\n            train, val, out = root / "train.jsonl", root / "val.jsonl", root / "out"\n            train.write_text("\\n".join(json.dumps(row(f"Training question {i}")) for i in range(4)), encoding="utf-8")\n            val.write_text(json.dumps(row("Held out question")), encoding="utf-8")\n            with socket.socket() as sock:\n                sock.bind(("127.0.0.1", 0))\n                port = sock.getsockname()[1]\n            env = dict(os.environ, USE_LIBUV="0", OMP_NUM_THREADS="1", MKL_NUM_THREADS="1", HF_HUB_OFFLINE="1")\n            command = [sys.executable, "-m", "torch.distributed.run", "--nnodes=1", "--nproc-per-node=2",\n                       "--master-addr=127.0.0.1", f"--master-port={port}",\n                       str(Path(__file__).with_name("train_tools.py")), "--smoke", "--train", str(train),\n                       "--validation", str(val), "--llm-dir", "unused-smoke-model", "--out", str(out),\n                       "--steps", "1", "--accum", "1", "--rank", "2", "--alpha", "4", "--max-length", "512"]\n            if os.name == "nt":\n                # PyTorch 2.5\'s Windows torchrun agent hardcodes a libuv\n                # TCPStore despite builds without libuv. Exercise the same\n                # rank/env contract with two directly launched workers.\n                workers = []\n                try:\n                    for rank in range(2):\n                        worker_env = dict(env, WORLD_SIZE="2", RANK=str(rank), LOCAL_RANK=str(rank),\n                                          MASTER_ADDR="127.0.0.1", MASTER_PORT=str(port))\n                        workers.append(subprocess.Popen([sys.executable] + command[7:], env=worker_env,\n                                                        stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True))\n                    for worker in workers:\n                        stdout, stderr = worker.communicate(timeout=120)\n                        self.assertEqual(worker.returncode, 0, stdout + stderr)\n                finally:\n                    for worker in workers:\n                        if worker.poll() is None:\n                            worker.kill()\n                            worker.communicate()\n            else:\n                result = subprocess.run(command, env=env, capture_output=True, text=True, timeout=120)\n                self.assertEqual(result.returncode, 0, result.stdout + result.stderr)\n            meta = json.loads((out / "adapter-step1.json").read_text())\n            self.assertEqual(meta["contract"]["world_size"], 2)\n            self.assertEqual(meta["lora"]["n_modules"], 7)\n            self.assertTrue((out / "checkpoint.pt").is_file())\n'}
SYSTEM_PROMPT = 'You are Ilaria, the local assistant for SwypikOS. Respond in English by default, unless the user explicitly requests another language. Never claim an action succeeded unless its tool result confirms success. Tool errors are failures, not evidence of completion.\n\nIf you need a tool, reply with EXACTLY one line in this form and nothing else — no words before it, no words after it, no explanation:\nCALL <tool>: <args>\n\nYou will then be given a line starting with "Tool:" carrying the result. Read it, then answer the user\'s question in plain language — do not just repeat the raw result verbatim if a full sentence reads better. If you do NOT need a tool, skip the CALL line entirely and answer directly in plain language.\n\nAvailable tools:\n- calc: <arithmetic expression> — exact arithmetic: + - * / ^, parentheses, sqrt(x)\n- time: (no args) — the current date and time, local and UTC, including day of week\n- convert: <N> <from-unit> to <to-unit> — unit conversion (km/mi, kg/lb, C/F, m/ft)\n\nExamples:\n\nUser: What is 12 times 7?\nAssistant: CALL calc: 12*7\nTool: 84\nAssistant: 12 times 7 is 84.\n\nUser: What is the capital of France?\nAssistant: The capital of France is Paris.'
WORK = Path("/content/ilaria-swypikos")
WORK.mkdir(parents=True, exist_ok=True)
manifest = {}
for relative, content in SOURCES.items():
    target = WORK / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(content, encoding="utf-8")
    manifest[relative] = hashlib.sha256(content.encode("utf-8")).hexdigest()
(WORK / "system_prompt.txt").write_text(SYSTEM_PROMPT, encoding="utf-8")
(WORK / "source_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
os.chdir(WORK)
print("Source bundle verified:", len(manifest), "files")

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
ROOT = Path("/content/drive/MyDrive/ilaria/swypikos-en")

In [ ]:
from huggingface_hub import HfApi, snapshot_download
MODEL_ID = "microsoft/bitnet-b1.58-2B-4T"
ROOT.mkdir(parents=True, exist_ok=True)
revision_file = ROOT / "base_revision.txt"
revision = revision_file.read_text().strip() if revision_file.exists() else HfApi().model_info(MODEL_ID).sha
revision_file.write_text(revision, encoding="utf-8")
MODEL = Path("/content/models") / revision
snapshot_download(MODEL_ID, revision=revision, local_dir=MODEL,
    allow_patterns=["*.json", "*.safetensors", "*.jinja", "*.model", "*.txt"])
print({"model": MODEL_ID, "revision": revision, "path": str(MODEL)})

In [ ]:
import base64, gzip, json, hashlib, subprocess, sys
from pathlib import Path
import torch
assert torch.cuda.is_available()
print('Actual GPU:', torch.cuda.get_device_name(0))
WORK = Path('/content/ilaria-swypikos')
ROOT = Path('/content/drive/MyDrive/ilaria/swypikos-en')
assert ROOT.is_dir()
MODEL = Path('/content/models/04c3b9ad9361b824064a1f25ea60a8be9599b127')
expected_sources = {"forge/tool_data.py":"757665542e531c9e9aad2da8f4c023d324f29c5d9bc8cf6faa7c6c4255e8a6cd","forge/train_tools.py":"a467ea3229ca98aa6554e0ed27b1c6326cd7e7a195d9524d5f719ab342fc8848"}
for name, expected in expected_sources.items():
    assert hashlib.sha256((WORK/name).read_bytes()).hexdigest() == expected, name
sys.path.insert(0,str(WORK))
import importlib
import forge.tool_data
importlib.reload(forge.tool_data)
from forge.tool_data import load_trajectories, validate_splits, encode_trajectory, ENCODING_VERSION
assert ENCODING_VERSION == 'assistant-header-split-v2'
DATA_V6 = ROOT/'datasets/project-v6'
RUN_V6 = ROOT/'runs/project-v6-200steps'
INIT_V6 = ROOT/'runs/drafting-v5-150steps/adapter-step150'
assert not RUN_V6.exists(), 'Output already exists: inspect instead of overwriting'
assert hashlib.sha256(Path(str(INIT_V6)+'.safetensors').read_bytes()).hexdigest() == '1a275f6a5541da2cc942d25c6bd7530fcfbb74c17c501cf0e49a1efbca4e8446'
DATA_V6.mkdir(parents=True,exist_ok=True)
payload = json.loads(gzip.decompress(base64.b64decode('...')))
for name, content in payload.items():
    target = DATA_V6/name
    if target.exists():
        assert target.read_text() == content
    else:
        target.write_text(content,encoding='utf-8')
train = load_trajectories(DATA_V6/'train.jsonl')
validation = load_trajectories(DATA_V6/'validation.jsonl')
print(validate_splits(train,validation))
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(str(MODEL),local_files_only=True)
lengths = [len(encode_trajectory(row,tok,2048)['input_ids']) for row in train+validation]
print('Preflight rows:',len(lengths),'max tokens:',max(lengths),'no truncation')
args_v6=[sys.executable,'-u','-m','forge.train_tools','--train',str(DATA_V6/'train.jsonl'),'--validation',str(DATA_V6/'validation.jsonl'),'--llm-dir',str(MODEL),'--out',str(RUN_V6),'--init-adapter',str(INIT_V6),'--languages','en','--steps','200','--batch','1','--accum','8','--max-length','2048','--rank','16','--alpha','32','--lr','0.00005','--gradient-checkpointing','--checkpoint-every','50','--log-every','25','--max-runtime-minutes','15']
with subprocess.Popen(args_v6,cwd=WORK,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1) as process:
    for line in process.stdout:
        print(line,end='',flush=True)
    rc_v6=process.wait()
assert rc_v6 == 0
(RUN_V6/'source_manifest.json').write_text(json.dumps(expected_sources,indent=2))
(RUN_V6/'dataset_manifest.json').write_text(payload['manifest.json'])
(RUN_V6/'environment.txt').write_text(subprocess.check_output([sys.executable,'-m','pip','freeze'],text=True))
print('PROJECT V6 COMPLETE:',RUN_V6,flush=True)


In [ ]:
assert rc_v6 == 0
from safetensors import safe_open
meta_v6=json.loads((RUN_V6/'adapter-step200.json').read_text())
assert meta_v6['contract']['encoding_version']=='assistant-header-split-v2'
assert meta_v6['contract']['initial_adapter']['weights_sha256']=='1a275f6a5541da2cc942d25c6bd7530fcfbb74c17c501cf0e49a1efbca4e8446'
with safe_open(str(RUN_V6/'adapter-step200.safetensors'),framework='pt',device='cpu') as sf:
    assert len(list(sf.keys()))==420
    for key in sf.keys():
        assert torch.isfinite(sf.get_tensor(key)).all()
artifacts_v6={n:hashlib.sha256((RUN_V6/n).read_bytes()).hexdigest() for n in ['checkpoint.pt','adapter-step200.json','adapter-step200.safetensors']}
(RUN_V6/'artifact_manifest.json').write_text(json.dumps(artifacts_v6,indent=2))
print('VERIFIED V6: 200 steps; 420 finite tensors; parent and encoding confirmed')
print(json.dumps(artifacts_v6))
import shutil
for suffix in ['json','safetensors']:
    shutil.copyfile(RUN_V6/('adapter-step200.'+suffix),Path('/content/ilaria-project-v6.'+suffix))
sys.path.insert(0,str(WORK/'forge/multimodal'))
import lora_bitlinear as lb
from forge.train_tools import initialize_adapter
probe_tok=AutoTokenizer.from_pretrained(str(MODEL),local_files_only=True)
probe_model=lb.load_frozen_base('offline',str(MODEL)).to('cuda')
lb.inject_lora(probe_model,r=16,alpha=32,dropout=0)
probe_model.to('cuda')
initialize_adapter(probe_model,str(RUN_V6/'adapter-step200'),lb,'offline',16,32)
probe_model.eval()
project_tasks=json.loads("[{\"id\":\"project-os\",\"prompt\":\"Evidence: SwypikOS is a Windows desktop shell; independent boot is not implemented. Can I install it as a standalone bootable operating system?\",\"expected_tool\":\"\",\"rubric\":\"No; currently requires Windows.\"},{\"id\":\"project-proof\",\"prompt\":\"Evidence: a Swyp candidate passed five supplied points, with no universal proof. Is it guaranteed correct for every float64 input?\",\"expected_tool\":\"\",\"rubric\":\"No; finite checks are not a universal proof.\"},{\"id\":\"project-static\",\"prompt\":\"Evidence: a generated Swyp draft passed parsing and type checks but was never run. Is the requested behavior verified?\",\"expected_tool\":\"\",\"rubric\":\"No; behavioral tests still required.\"},{\"id\":\"project-budget\",\"prompt\":\"Evidence: the synthesis search exhausted its 4000-candidate limit. Does this prove no solution exists?\",\"expected_tool\":\"\",\"rubric\":\"No; bounded search failure only.\"},{\"id\":\"project-adapter\",\"prompt\":\"Evidence: an adapter implements Name/Match/Execute, while the chat server accepts Name/Describe/Call. Is registration of the old adapter alone enough?\",\"expected_tool\":\"\",\"rubric\":\"No; requires a compatible chat adapter and registration.\"},{\"id\":\"project-graph\",\"prompt\":\"Evidence: the returned graph contains numeric inputs, constants and arithmetic operations. Is this a quantum state?\",\"expected_tool\":\"\",\"rubric\":\"No; numerical operation graph.\"},{\"id\":\"project-timeout\",\"prompt\":\"Evidence: the service timed out and returned no action result. Tell the user the status without assuming success.\",\"expected_tool\":\"\",\"rubric\":\"Timeout; completion unconfirmed.\"},{\"id\":\"project-arrays\",\"prompt\":\"Evidence: this Swyp release has scalar values but no arrays or imports. Give me a working array-import API.\",\"expected_tool\":\"\",\"rubric\":\"Explicitly identify unsupported capability; no invented API.\"},{\"id\":\"grounded-date\",\"prompt\":\"Draft a two-sentence notice: the rehearsal is postponed. No replacement date has been decided. Do not invent one.\",\"expected_tool\":\"\",\"rubric\":\"Two sentences; no invented date.\"},{\"id\":\"grounded-price\",\"prompt\":\"Write a short product announcement using only: a new desk lamp is available. Its price has not been supplied.\",\"expected_tool\":\"\",\"rubric\":\"No invented price, discount or availability details.\"},{\"id\":\"grounded-name\",\"prompt\":\"Draft a thank-you note to the person who checked the report. Their name is unknown.\",\"expected_tool\":\"\",\"rubric\":\"No invented name; thanks only.\"},{\"id\":\"grounded-deadline\",\"prompt\":\"Write an update: registration remains open. No closing date has been announced.\",\"expected_tool\":\"\",\"rubric\":\"No invented deadline.\"},{\"id\":\"swyp-affine\",\"prompt\":\"Swyp uses fn name(x: number) -> number { return expression; }. Return only a complete program with predict and main; main must print predict(arg(0)).\\nImplement y = 31*x - 8.\",\"expected_tool\":\"\",\"rubric\":\"Compiles and yields 31*x-8 on unseen inputs.\"},{\"id\":\"swyp-square\",\"prompt\":\"Swyp uses fn name(x: number) -> number { return expression; }. Return only a complete program with predict and main; main must print predict(arg(0)).\\nImplement the square of the numeric input.\",\"expected_tool\":\"\",\"rubric\":\"Compiles and yields x*x.\"},{\"id\":\"swyp-negative\",\"prompt\":\"Swyp uses fn name(x: number) -> number { return expression; }. Return only a complete program with predict and main; main must print predict(arg(0)).\\nReturn the negative of the numeric input.\",\"expected_tool\":\"\",\"rubric\":\"Compiles and yields -x.\"},{\"id\":\"swyp-constant\",\"prompt\":\"Swyp uses fn name(x: number) -> number { return expression; }. Return only a complete program with predict and main; main must print predict(arg(0)).\\nReturn the constant 42 for every numeric input.\",\"expected_tool\":\"\",\"rubric\":\"Compiles and yields 42.\"}]")
probe_system=train[0]['messages'][0]['content']
answers_v6=[]
for task in project_tasks:
    prompt='System: '+probe_system+'<|eot_id|>User: '+task['prompt']+'<|eot_id|>Assistant: '
    ids=probe_tok(prompt,return_tensors='pt',add_special_tokens=False).input_ids.to('cuda')
    with torch.no_grad(),torch.autocast('cuda',dtype=torch.bfloat16):
        generated=probe_model.generate(ids,attention_mask=torch.ones_like(ids),do_sample=False,max_new_tokens=256,pad_token_id=probe_tok.eos_token_id,eos_token_id=[probe_tok.eos_token_id,probe_tok.convert_tokens_to_ids('<|eot_id|>')])
    row={**task,'generation':probe_tok.decode(generated[0,ids.shape[1]:],skip_special_tokens=True)}
    answers_v6.append(row)
    print('V6ANSWER '+json.dumps(row,ensure_ascii=False),flush=True)
(RUN_V6/'python-project-evaluation.json').write_text(json.dumps(answers_v6,ensure_ascii=False,indent=2))
del probe_model
import gc
gc.collect()
torch.cuda.empty_cache()
print('V6 PYTHON CHECK COMPLETE; Go verification remains separate')
from google.colab import files
files.download('/content/ilaria-project-v6.json')
files.download('/content/ilaria-project-v6.safetensors')
